第1步，筛选点云：以下代码实现只要输入截止日期，它能倒退两年筛选点云大于4个的股票出来，每个点云具有60个交易日的数据。

In [ ]:
import os
import pandas as pd
from datetime import datetime
import logging

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class StockDataProcessor:
    """股票数据处理类"""
    
    def __init__(self, source_dir, target_base_dir, end_date):
        """
        初始化处理器
        
        Args:
            source_dir: 源数据目录
            target_base_dir: 目标基础目录
            end_date: 截止日期 (pd.Timestamp)
        """
        self.source_dir = source_dir
        self.target_base_dir = target_base_dir
        self.end_date = end_date
        self.start_date = end_date - pd.DateOffset(years=2)
        
        # 创建带日期的目标文件夹
        self.date_folder = end_date.strftime('%Y%m%d')
        self.target_dir = os.path.join(target_base_dir, self.date_folder)
        
        # 统计信息
        self.processed_count = 0
        self.discarded_count = 0
        self.error_count = 0
        
        # 配置参数
        self.points_per_cloud = 60
        self.min_clouds = 4
        self.max_clouds = 8
        self.min_rows = self.min_clouds * self.points_per_cloud
        
    def _ensure_target_directory(self):
        """确保目标目录存在"""
        try:
            os.makedirs(self.target_dir, exist_ok=True)
            logger.info(f"目标目录已创建/存在: {self.target_dir}")
            return True
        except Exception as e:
            logger.error(f"创建目标目录失败: {e}")
            return False
    
    def _clean_stock_code(self, filename):
        """清理股票代码"""
        return filename.replace('.csv', '').replace('.', '')
    
    def _validate_dataframe(self, df, filename):
        """验证DataFrame是否包含必要列"""
        if 'EventDate' not in df.columns:
            logger.warning(f"跳过 {filename}: 缺少 'EventDate' 列")
            return False
        return True
    
    def _filter_by_date(self, df):
        """按日期范围过滤数据"""
        df['EventDate'] = pd.to_datetime(df['EventDate'])
        mask = (df['EventDate'] >= self.start_date) & (df['EventDate'] <= self.end_date)
        df_filtered = df.loc[mask].copy()
        df_filtered.sort_values(by='EventDate', ascending=False, inplace=True)
        df_filtered.reset_index(drop=True, inplace=True)
        return df_filtered
    
    def _calculate_clouds(self, total_rows):
        """计算点云数量"""
        if total_rows < self.min_rows:
            return 0  # 数据不足
        
        num_clouds = total_rows // self.points_per_cloud
        return min(num_clouds, self.max_clouds)
    
    def _generate_cloud_metadata(self, df_final, num_clouds, stock_code):
        """生成点云ID和名称"""
        cloud_ids = []
        cloud_names = []
        
        for i in range(num_clouds):
            start_idx = i * self.points_per_cloud
            end_idx = (i + 1) * self.points_per_cloud
            
            cloud_slice = df_final.iloc[start_idx:end_idx]
            latest_date = cloud_slice['EventDate'].iloc[0]
            date_str = latest_date.strftime('%Y%m%d')
            
            cloud_name = f"{date_str}{stock_code}"
            
            cloud_ids.extend([i] * self.points_per_cloud)
            cloud_names.extend([cloud_name] * self.points_per_cloud)
        
        return cloud_ids, cloud_names
    
    def process_single_file(self, filename):
        """
        处理单个CSV文件
        
        Returns:
            bool: 处理是否成功
        """
        file_path = os.path.join(self.source_dir, filename)
        
        try:
            # 读取数据
            df = pd.read_csv(file_path)
            
            # 验证数据
            if not self._validate_dataframe(df, filename):
                return False
            
            # 日期过滤和排序
            df_filtered = self._filter_by_date(df)
            total_rows = len(df_filtered)
            
            # 计算点云数量
            num_clouds = self._calculate_clouds(total_rows)
            
            if num_clouds == 0:
                logger.debug(f"丢弃 {filename}: 数据不足 (共{total_rows}行)")
                self.discarded_count += 1
                return False
            
            # 选择保留的数据
            rows_to_keep = num_clouds * self.points_per_cloud
            df_final = df_filtered.iloc[:rows_to_keep].copy()
            
            # 生成点云元数据
            stock_code = self._clean_stock_code(filename)
            cloud_ids, cloud_names = self._generate_cloud_metadata(
                df_final, num_clouds, stock_code
            )
            
            # 添加列
            df_final['PointCloudID'] = cloud_ids
            df_final['PointCloudName'] = cloud_names
            
            # 保存文件
            target_path = os.path.join(self.target_dir, filename)
            df_final.to_csv(target_path, index=False)
            
            self.processed_count += 1
            logger.info(f"已处理: {filename} (点云数: {num_clouds})")
            return True
            
        except Exception as e:
            logger.error(f"处理 {filename} 时出错: {e}")
            self.error_count += 1
            return False
    
    def process_all_files(self):
        """处理所有文件"""
        # 确保目标目录存在
        if not self._ensure_target_directory():
            return
        
        # 获取所有CSV文件
        files = [f for f in os.listdir(self.source_dir) if f.endswith('.csv')]
        logger.info(f"在 {self.source_dir} 中找到 {len(files)} 个CSV文件")
        
        if not files:
            logger.warning("未找到任何CSV文件")
            return
        
        # 处理每个文件
        for filename in files:
            self.process_single_file(filename)
        
        # 输出统计信息
        self._print_summary()
    
    def _print_summary(self):
        """打印处理摘要"""
        logger.info("=" * 50)
        logger.info("处理完成!")
        logger.info(f"时间范围: {self.start_date.strftime('%Y-%m-%d')} 至 {self.end_date.strftime('%Y-%m-%d')}")
        logger.info(f"目标文件夹: {self.date_folder}")
        logger.info(f"成功处理: {self.processed_count} 个股票")
        logger.info(f"数据不足丢弃: {self.discarded_count} 个股票")
        logger.info(f"处理错误: {self.error_count} 个股票")
        logger.info("=" * 50)


def get_user_input():
    """获取用户输入"""
    print("\n" + "=" * 50)
    print("股票数据点云生成工具 v2.0")
    print("=" * 50)
    
    while True:
        input_date_str = input("\n请输入截止日期 (格式 YYYY-MM-DD, 例如 2023-12-31): ").strip()
        
        if not input_date_str:
            print("输入不能为空，请重新输入")
            continue
        
        try:
            end_date = pd.to_datetime(input_date_str)
            return end_date
        except Exception as e:
            print(f"日期格式错误: {e}")
            print("请使用 YYYY-MM-DD 格式，例如: 2023-12-31")


def main():
    """主函数"""
    # 配置路径
    source_directory = r"D:\量化\我自己的实验\3rd test\stock"
    target_base_directory = r"D:\量化\我自己的实验\3rd test"
    
    # 获取用户输入
    end_date = get_user_input()
    
    # 显示配置信息
    start_date = end_date - pd.DateOffset(years=2)
    date_folder = end_date.strftime('%Y%m%d')
    
    print("\n" + "-" * 50)
    print(f"时间范围: {start_date.strftime('%Y-%m-%d')} 至 {end_date.strftime('%Y-%m-%d')}")
    print(f"数据将保存至: {os.path.join(target_base_directory, date_folder)}")
    print("-" * 50)
    
    # 确认执行
    confirm = input("\n是否继续? (y/n): ").strip().lower()
    if confirm != 'y':
        print("操作已取消")
        return
    
    # 创建处理器并执行
    try:
        processor = StockDataProcessor(
            source_dir=source_directory,
            target_base_dir=target_base_directory,
            end_date=end_date
        )
        processor.process_all_files()
        
    except Exception as e:
        logger.error(f"程序执行失败: {e}")
        raise


if __name__ == "__main__":
    main()

第2步，切割点云：我有一个文件夹，里面有5027支股票从当前日（2024年6月28日）往前2年的数据，每个csv内都有PointCloudName这一列，请你帮我写一个python程序，读取文件夹中全部的csv，把每个csv中的数据按照PointCloudName这一列切割，把值相同的数据行分成一个新的csv，每个新的csv文件名就以该值来命名。把所有新的csv另存到一个新的文件夹中。

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import time
import traceback

def split_csv_by_pointcloudname(input_folder, output_folder):
    """
    将输入文件夹中的所有CSV文件按照PointCloudName列的值进行分割
    每个不同的值保存为一个新的CSV文件
    """
    # 创建输出文件夹
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # 获取所有CSV文件
    input_path = Path(input_folder)
    csv_files = list(input_path.glob("*.csv"))
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("开始处理...")
    
    # 记录处理统计
    total_files = len(csv_files)
    processed_files = 0
    total_splits = 0
    
    # 用于跟踪所有不同的PointCloudName值
    all_pointcloud_names = set()
    
    # 处理每个CSV文件
    for csv_file in csv_files:
        try:
            start_time = time.time()
            print(f"\n处理文件: {csv_file.name}")
            
            # 读取CSV文件
            df = pd.read_csv(csv_file)
            
            # 检查是否包含PointCloudName列
            if 'PointCloudName' not in df.columns:
                print(f"警告: 文件 {csv_file.name} 不包含PointCloudName列，跳过")
                continue
            
            # 获取唯一的PointCloudName值
            unique_names = df['PointCloudName'].dropna().unique()
            
            # 更新全局集合
            all_pointcloud_names.update(unique_names)
            
            print(f"  找到 {len(unique_names)} 个不同的PointCloudName值")
            
            # 按照PointCloudName分组并保存
            for name in unique_names:
                try:
                    # 筛选数据
                    subset = df[df['PointCloudName'] == name]
                    
                    if not subset.empty:
                        # 创建安全的文件名（处理特殊字符）
                        safe_name = str(name).replace('/', '_').replace('\\', '_').replace(':', '_')
                        
                        # 构建输出文件名
                        output_filename = f"{safe_name}.csv"
                        output_filepath = output_path / output_filename
                        
                        # 如果文件已存在，追加数据
                        if output_filepath.exists():
                            existing_df = pd.read_csv(output_filepath)
                            combined_df = pd.concat([existing_df, subset], ignore_index=True)
                            combined_df.to_csv(output_filepath, index=False)
                            mode = "追加到"
                        else:
                            subset.to_csv(output_filepath, index=False)
                            mode = "创建"
                        
                        total_splits += 1
                        
                except Exception as e:
                    print(f"  处理 PointCloudName='{name}' 时出错: {str(e)}")
                    continue
            
            processed_files += 1
            elapsed_time = time.time() - start_time
            print(f"  完成处理 {csv_file.name}，耗时: {elapsed_time:.2f}秒")
            
            # 每处理100个文件打印一次进度
            if processed_files % 100 == 0:
                print(f"\n已处理 {processed_files}/{total_files} 个文件")
                
        except Exception as e:
            print(f"处理文件 {csv_file.name} 时出错: {str(e)}")
            traceback.print_exc()
            continue
    
    # 生成处理报告
    print("\n" + "="*50)
    print("处理完成！")
    print("="*50)
    print(f"处理了 {processed_files}/{total_files} 个文件")
    print(f"创建了 {total_splits} 个分割文件")
    print(f"总共找到 {len(all_pointcloud_names)} 个不同的PointCloudName值")
    print(f"输出文件夹: {output_path.absolute()}")
    
    # 保存PointCloudName值的列表
    if all_pointcloud_names:
        names_list_file = output_path / "all_pointcloud_names.txt"
        with open(names_list_file, 'w', encoding='utf-8') as f:
            for name in sorted(all_pointcloud_names):
                f.write(f"{name}\n")
        print(f"PointCloudName值列表已保存到: {names_list_file}")

def main():
    # 配置输入和输出文件夹路径
    input_folder = r"D:\量化\我自己的实验\3rd test\20240630"  # 修改为您的输入文件夹路径
    output_folder = r"D:\量化\我自己的实验\3rd test\split_by_pointcloudname"  # 输出文件夹路径
    
    # 验证输入文件夹是否存在
    if not os.path.exists(input_folder):
        print(f"错误: 输入文件夹 '{input_folder}' 不存在")
        print("请修改 input_folder 变量为正确的路径")
        return
    
    print("股票数据分割程序")
    print("="*50)
    print(f"输入文件夹: {os.path.abspath(input_folder)}")
    print(f"输出文件夹: {os.path.abspath(output_folder)}")
    print("="*50)
    
    # 确认是否继续
    response = input("是否开始处理？(y/n): ")
    if response.lower() != 'y':
        print("程序已取消")
        return
    
    # 执行分割
    split_csv_by_pointcloudname(input_folder, output_folder)

if __name__ == "__main__":
    main()

第3步，归一化：我有一个文件夹，里面有39318个csv，帮我写一个python程序，读取该文件夹中的每一个csv，将从第C列到第L列的每一列数据分别按列归一化。然后把结果存到一个新建的同名的csv中，并且把所有新生成的csv存到一个新的文件夹中。耗时约4分钟

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def normalize_columns_C_to_L(csv_file, output_folder):
    """
    读取CSV文件，对C列到L列进行归一化处理
    """
    try:
        # 读取CSV文件
        df = pd.read_csv(csv_file, low_memory=False)
        
        # 获取列名
        columns = df.columns.tolist()
        
        if len(columns) < 12:  # C列是第3列，索引从0开始，所以需要至少12列
            print(f"文件 {csv_file.name} 列数不足，跳过处理")
            return None
        
        # 确定要归一化的列范围（C列到L列）
        # C列是第3列（索引2），L列是第12列（索引11）
        # 因为CSV列索引从0开始：A=0, B=1, C=2, ..., L=11
        start_idx = 2  # C列索引
        end_idx = min(11, len(columns) - 1)  # L列索引，确保不超出范围
        
        # 获取要归一化的列名
        cols_to_normalize = columns[start_idx:end_idx + 1]
        
        if not cols_to_normalize:
            print(f"文件 {csv_file.name} 没有C-L列，跳过处理")
            return df
        
        # 创建数据的副本用于归一化
        df_normalized = df.copy()
        
        # 对每一列进行归一化
        for col in cols_to_normalize:
            try:
                # 检查列是否为数值类型
                if pd.api.types.is_numeric_dtype(df_normalized[col]):
                    col_data = df_normalized[col]
                    
                    # 计算最小值和最大值
                    col_min = col_data.min()
                    col_max = col_data.max()
                    
                    # 避免除以零
                    if col_max - col_min != 0:
                        # 进行归一化： (x - min) / (max - min)
                        df_normalized[col] = (col_data - col_min) / (col_max - col_min)
                    elif col_max == col_min and not np.isnan(col_max):
                        # 如果所有值都相同，归一化为0或1
                        df_normalized[col] = 0.0 if col_max == 0 else 1.0
                
            except Exception as e:
                print(f"  列 '{col}' 归一化失败: {str(e)}")
                continue
        
        # 保存到新的CSV文件
        output_file = output_folder / csv_file.name
        df_normalized.to_csv(output_file, index=False)
        
        return df_normalized
        
    except Exception as e:
        print(f"处理文件 {csv_file.name} 时出错: {str(e)}")
        return None

def process_all_csvs(input_folder, output_folder, batch_size=1000):
    """
    处理输入文件夹中的所有CSV文件
    """
    # 创建输出文件夹
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # 获取所有CSV文件
    input_path = Path(input_folder)
    csv_files = list(input_path.glob("*.csv"))
    
    if not csv_files:
        print(f"在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print(f"开始处理，输出到: {output_path.absolute()}")
    print("-" * 60)
    
    # 统计信息
    processed_count = 0
    failed_count = 0
    start_time = time.time()
    
    # 分批处理以提高性能
    for i in tqdm(range(0, len(csv_files), batch_size), desc="处理进度"):
        batch_files = csv_files[i:i + batch_size]
        batch_start = time.time()
        
        for csv_file in batch_files:
            try:
                result = normalize_columns_C_to_L(csv_file, output_path)
                if result is not None:
                    processed_count += 1
                else:
                    failed_count += 1
                    
                # 每处理1000个文件输出一次进度
                if processed_count % 1000 == 0:
                    elapsed = time.time() - start_time
                    print(f"\n已处理 {processed_count} 个文件，失败 {failed_count} 个")
                    print(f"运行时间: {elapsed:.2f}秒")
                    print(f"平均速度: {processed_count/elapsed:.2f} 文件/秒")
                    
            except Exception as e:
                print(f"\n处理文件 {csv_file.name} 时发生严重错误: {str(e)}")
                failed_count += 1
                continue
        
        batch_time = time.time() - batch_start
        if i > 0:
            estimated_total = (len(csv_files) / processed_count) * (time.time() - start_time)
            print(f"\n批次完成: {i+batch_size}/{len(csv_files)}，本批次耗时: {batch_time:.2f}秒")
            print(f"预计剩余时间: {(estimated_total - (time.time() - start_time))/60:.2f}分钟")
    
    # 输出最终统计
    total_time = time.time() - start_time
    print("\n" + "=" * 60)
    print("处理完成!")
    print("=" * 60)
    print(f"总计文件: {len(csv_files)}")
    print(f"成功处理: {processed_count}")
    print(f"处理失败: {failed_count}")
    print(f"总耗时: {total_time:.2f}秒 ({total_time/60:.2f}分钟)")
    print(f"处理速度: {processed_count/total_time:.2f} 文件/秒")
    print(f"输出文件夹: {output_path.absolute()}")

def main():
    # 配置路径
    input_folder = r"D:\量化\我自己的实验\3rd test\split_by_pointcloudname"  # 修改为您的输入文件夹路径
    output_folder = r"D:\量化\我自己的实验\3rd test\normalized_data"  # 输出文件夹路径


    
    # 验证输入文件夹
    if not os.path.exists(input_folder):
        print(f"错误: 输入文件夹 '{input_folder}' 不存在")
        print("请修改 input_folder 变量为正确的路径")
        return
    
    # 显示配置信息
    print("CSV文件归一化处理程序")
    print("=" * 60)
    print("功能: 对每个CSV文件的C列到L列进行归一化处理")
    print(f"输入文件夹: {os.path.abspath(input_folder)}")
    print(f"输出文件夹: {os.path.abspath(output_folder)}")
    print("=" * 60)
    
    # 内存使用警告（处理大量文件时）
    total_size = sum(f.stat().st_size for f in Path(input_folder).glob("*.csv"))
    print(f"预估总数据量: {total_size/1024/1024/1024:.2f} GB")
    
    if total_size > 10 * 1024**3:  # 大于10GB
        print("警告: 数据量较大，请确保有足够内存")
        print("建议分批处理或增加内存")
    
    # 确认开始
    response = input("\n是否开始处理？(y/n): ")
    if response.lower() != 'y':
        print("程序已取消")
        return
    
    # 开始处理
    process_all_csvs(input_folder, output_folder)

if __name__ == "__main__":
    main()

第4步，持续同调：我有一个文件夹，里面有三万多个csv，帮我写一个python程序，读取该文件夹中每一个csv中的4列：money, volume, high, close, 生成一个点云，使用gudhi, 然后执行以下代码，并把结果保存在一个与csv同名的txt文件中，然后把所有新生成的txt文件保存在一个新的文件夹中，约70分钟：
%2. 计算持续同调 =======================================================
%构建Rips复形
rips_complex = gd.RipsComplex(points=points, max_edge_length=3.0)
simplex_tree = rips_complex.create_simplex_tree(max_dimension=3)
%计算持续同调（维度0,1,2）
persistence = simplex_tree.persistence()

In [ ]:
import os
import pandas as pd
import numpy as np
import gudhi as gd
from pathlib import Path
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def calculate_persistence_for_csv(csv_file, output_folder):
    """
    读取CSV文件，提取指定4列，计算持续同调，保存结果
    """
    try:
        # 读取CSV文件
        df = pd.read_csv(csv_file, low_memory=False)
        
        # 检查所需的列是否存在
        required_columns = ['money', 'volume', 'high', 'close']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            print(f"文件 {csv_file.name} 缺少列: {missing_columns}")
            return False
        
        # 提取所需的4列数据
        points_data = df[required_columns].values
        
        # 检查数据是否有效（没有NaN或无穷大值）
        if np.isnan(points_data).any() or np.isinf(points_data).any():
            # 移除无效数据点
            points_data = points_data[~np.isnan(points_data).any(axis=1)]
            points_data = points_data[~np.isinf(points_data).any(axis=1)]
            
            if len(points_data) == 0:
                print(f"文件 {csv_file.name} 没有有效数据点")
                return False
        
        # 检查是否有足够的数据点（至少需要2个点）
        if len(points_data) < 2:
            print(f"文件 {csv_file.name} 数据点不足（{len(points_data)}个）")
            return False
        
        # 可选：对数据进行标准化，使计算更稳定
        # 方法1：归一化到[0,1]
        # points_data = (points_data - points_data.min(axis=0)) / (points_data.max(axis=0) - points_data.min(axis=0) + 1e-10)
        
        # 方法2：Z-score标准化（推荐）
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        points_data = scaler.fit_transform(points_data)
        
        # 1. 构建点云
        points = points_data
        
        # 2. 计算持续同调
        # 构建Rips复形
        try:
            rips_complex = gd.RipsComplex(points=points, max_edge_length=3.0)
            simplex_tree = rips_complex.create_simplex_tree(max_dimension=3)
            
            # 计算持续同调（维度0,1,2）
            persistence = simplex_tree.persistence()
            
            # 计算持续同调统计信息
            persistence_stats = {
                'num_points': len(points),
                'dimensions': [0, 1, 2],
                'betti_numbers': [simplex_tree.betti_numbers()[i] for i in range(3)]
            }
            
        except Exception as e:
            print(f"GUDHI计算失败 {csv_file.name}: {str(e)}")
            return False
        
        # 3. 保存结果到TXT文件
        output_file = output_folder / f"{csv_file.stem}.txt"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            # 写入文件基本信息
            f.write(f"原始文件: {csv_file.name}\n")
            f.write(f"处理时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"数据点数: {persistence_stats['num_points']}\n")
            f.write(f"Betti数 (0,1,2维): {persistence_stats['betti_numbers']}\n")
            f.write("-" * 60 + "\n\n")
            
            # 写入持续同调结果
            f.write("持续同调结果 (格式: (维度, (出生, 死亡))):\n")
            f.write("=" * 60 + "\n")
            
            for dim, (birth, death) in persistence:
                if dim <= 2:  # 只保存0,1,2维的结果
                    if death == float('inf'):
                        death_str = "inf"
                    else:
                        death_str = f"{death:.6f}"
                    
                    f.write(f"({dim}, ({birth:.6f}, {death_str}))\n")
            
            f.write("\n" + "=" * 60 + "\n\n")
            
            # 按维度分组显示
            f.write("按维度分组:\n")
            f.write("-" * 60 + "\n")
            
            for dim in range(3):
                dim_persistence = [(b, d) for d, (b, _) in enumerate(persistence) if d == dim]
                f.write(f"\n维度 {dim} (共 {len(dim_persistence)} 个特征):\n")
                
                # 获取该维度的所有持续同调对
                dim_pairs = [(birth, death) for d, (birth, death) in persistence if d == dim]
                for birth, death in dim_pairs:
                    if death == float('inf'):
                        death_str = "inf"
                    else:
                        death_str = f"{death:.6f}"
                    f.write(f"  出生: {birth:.6f}, 死亡: {death_str}\n")
            
            f.write("\n" + "=" * 60 + "\n\n")
            
            # 保存原始数据摘要
            f.write("数据摘要:\n")
            f.write("-" * 60 + "\n")
            for i, col in enumerate(required_columns):
                col_data = df[col].dropna()
                f.write(f"{col}: 均值={col_data.mean():.4f}, 标准差={col_data.std():.4f}, "
                       f"最小值={col_data.min():.4f}, 最大值={col_data.max():.4f}\n")
        
        return True
        
    except Exception as e:
        print(f"处理文件 {csv_file.name} 时出错: {str(e)}")
        return False

def process_all_csvs_with_persistence(input_folder, output_folder, max_edge_length=3.0, max_dimension=3):
    """
    处理输入文件夹中的所有CSV文件
    """
    # 创建输出文件夹
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # 获取所有CSV文件
    input_path = Path(input_folder)
    csv_files = list(input_path.glob("*.csv"))
    
    if not csv_files:
        print(f"在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print(f"开始处理持续同调计算，输出到: {output_path.absolute()}")
    print(f"参数: max_edge_length={max_edge_length}, max_dimension={max_dimension}")
    print("-" * 80)
    
    # 统计信息
    processed_count = 0
    failed_count = 0
    start_time = time.time()
    
    # 创建进度条
    pbar = tqdm(csv_files, desc="计算持续同调", unit="文件")
    
    # 结果摘要文件
    summary_file = output_path / "processing_summary.txt"
    
    for csv_file in pbar:
        try:
            # 更新进度条描述
            pbar.set_postfix_str(csv_file.name[:30])
            
            # 计算持续同调
            success = calculate_persistence_for_csv(csv_file, output_path)
            
            if success:
                processed_count += 1
            else:
                failed_count += 1
            
            # 每处理100个文件更新一次进度
            if processed_count % 100 == 0:
                elapsed = time.time() - start_time
                avg_time = elapsed / processed_count if processed_count > 0 else 0
                pbar.set_postfix_str(f"成功:{processed_count}, 失败:{failed_count}, 平均:{avg_time:.2f}s/文件")
                
        except Exception as e:
            print(f"\n处理文件 {csv_file.name} 时发生严重错误: {str(e)}")
            failed_count += 1
            continue
    
    # 输出最终统计
    total_time = time.time() - start_time
    
    # 保存处理摘要
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("持续同调计算处理摘要\n")
        f.write("=" * 80 + "\n")
        f.write(f"处理时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"输入文件夹: {input_path.absolute()}\n")
        f.write(f"输出文件夹: {output_path.absolute()}\n")
        f.write(f"总文件数: {len(csv_files)}\n")
        f.write(f"成功处理: {processed_count}\n")
        f.write(f"处理失败: {failed_count}\n")
        f.write(f"总耗时: {total_time:.2f}秒 ({total_time/60:.2f}分钟)\n")
        f.write(f"平均处理时间: {total_time/max(processed_count, 1):.2f}秒/文件\n")
        f.write(f"参数设置: max_edge_length={max_edge_length}, max_dimension={max_dimension}\n")
    
    print("\n" + "=" * 80)
    print("处理完成!")
    print("=" * 80)
    print(f"总计文件: {len(csv_files)}")
    print(f"成功处理: {processed_count}")
    print(f"处理失败: {failed_count}")
    print(f"总耗时: {total_time:.2f}秒 ({total_time/60:.2f}分钟)")
    print(f"处理速度: {processed_count/total_time:.2f} 文件/秒")
    print(f"输出文件夹: {output_path.absolute()}")
    print(f"处理摘要已保存到: {summary_file}")

def main():
    """
    主函数
    """
    # 配置路径
    input_folder = r"D:\量化\我自己的实验\3rd test\normalized_data"  # 修改为您的输入文件夹路径
    output_folder = r"D:\量化\我自己的实验\3rd test\persistence_results"  # 输出文件夹路径

    
    # GUDHI参数
    max_edge_length = 3.0  # Rips复形的最大边长
    max_dimension = 3      # 计算的最大维度
    
    # 验证输入文件夹
    if not os.path.exists(input_folder):
        print(f"错误: 输入文件夹 '{input_folder}' 不存在")
        print("请修改 input_folder 变量为正确的路径")
        return
    
    # 显示配置信息
    print("持续同调计算程序")
    print("=" * 80)
    print("功能: 读取每个CSV文件的money, volume, high, close四列")
    print("      计算持续同调，保存结果到TXT文件")
    print(f"输入文件夹: {os.path.abspath(input_folder)}")
    print(f"输出文件夹: {os.path.abspath(output_folder)}")
    print(f"GUDHI参数: max_edge_length={max_edge_length}, max_dimension={max_dimension}")
    print("=" * 80)
    
    # 检查GUDHI是否安装
    try:
        import gudhi
        print(f"GUDHI版本: {gudhi.__version__}")
    except ImportError:
        print("错误: 未安装GUDHI库")
        print("请运行: pip install gudhi")
        return
    
    # 显示数据量估算
    csv_files = list(Path(input_folder).glob("*.csv"))
    if csv_files:
        print(f"找到 {len(csv_files)} 个CSV文件")
        
        # 估算处理时间（假设平均每个文件1000行数据）
        estimated_time = len(csv_files) * 2  # 假设每个文件2秒
        print(f"预估处理时间: {estimated_time/60:.1f}分钟 ({estimated_time/3600:.1f}小时)")
    
    # 确认开始
    response = input("\n是否开始处理？(y/n): ")
    if response.lower() != 'y':
        print("程序已取消")
        return
    
    # 开始处理
    process_all_csvs_with_persistence(
        input_folder, 
        output_folder,
        max_edge_length=max_edge_length,
        max_dimension=max_dimension
    )

if __name__ == "__main__":
    main()

第5步，分类点云，第6步，计算第一类点云与第二类点云的0维与1维瓶颈距离，第7步，若一个第一类点云能够选出N个或以上相似的第二类点云，就把它记录下来，相似的判断标准是两个瓶颈距离都小于0.1。本轮实验采取N=5. 第8步，一个第一类点云被记录，则同时记录N个与他相似的点云，排序按1维瓶颈距离从小到大。本段代码可以实现选择从第M个到第N个点云切片，方便分开操作。第9步，记录所有530个点云切片。耗时10分钟。

In [ ]:
import os
import re
import math
import time
import warnings
from pathlib import Path
from multiprocessing import cpu_count

import gudhi as gd
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

warnings.filterwarnings('ignore')


class PersistenceDiagramProcessor:
    def __init__(self, data_folder, current_date, num_workers=None):
        """
        初始化处理器

        Args:
            data_folder: 包含所有txt点云文件的文件夹路径
            current_date: 当前日期 (格式: YYYYMMDD)
            num_workers: 并行工作进程数，默认为CPU核心数减1
        """
        self.data_folder = Path(data_folder)
        self.current_date = current_date
        self.txt_files = list(self.data_folder.glob("*.txt"))

        if num_workers is None:
            self.num_workers = max(1, cpu_count() - 1)
        else:
            self.num_workers = max(1, min(num_workers, cpu_count()))

        print(f"找到 {len(self.txt_files)} 个点云文件")
        print(f"使用 {self.num_workers} 个并行进程")

        # 解析文件名中的日期和股票代码
        self.file_info = self._parse_filenames()

        # 分类文件：当前日期和其他日期
        self.current_date_files = []
        self.other_date_files = []
        self._classify_files_by_date()

        print(f"当前日期 ({self.current_date}) 文件数: {len(self.current_date_files)}")
        print(f"其他日期文件数: {len(self.other_date_files)}")

    def _parse_filenames(self):
        """
        解析文件名，提取日期和股票代码
        文件名格式: YYYYMMDD_stockcode.txt 或 YYYYMMDDstockcode.txt
        """
        file_info = {}
        pattern = re.compile(r'(\d{8})[_-]?(\w+)\.txt')

        for txt_file in self.txt_files:
            match = pattern.fullmatch(txt_file.name)
            if match:
                date_str, stock_code = match.groups()
                file_info[txt_file.name] = {
                    'path': txt_file,
                    'date': date_str,
                    'stock_code': stock_code,
                    'full_date': f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
                }
            else:
                file_info[txt_file.name] = {
                    'path': txt_file,
                    'date': 'unknown',
                    'stock_code': 'unknown',
                    'full_date': 'unknown'
                }

        return file_info

    def _classify_files_by_date(self):
        """将文件分为当前日期和其他日期两类"""
        for filename, info in self.file_info.items():
            if info['date'] == self.current_date:
                self.current_date_files.append(filename)
            else:
                self.other_date_files.append(filename)

    def get_subset_files_by_range(self, start_idx, end_idx):
        """
        获取指定范围的文件列表（索引从0开始）

        Args:
            start_idx: 起始索引（包含）
            end_idx: 结束索引（不包含）

        Returns:
            指定范围内的文件名列表
        """
        total_files = len(self.current_date_files)

        # 边界检查
        if start_idx < 0:
            start_idx = 0

        if end_idx > total_files:
            end_idx = total_files

        if start_idx >= total_files or start_idx >= end_idx:
            print(f"错误: 索引范围 [{start_idx}, {end_idx}) 无效")
            print(f"当前日期文件总数: {total_files}")
            return []

        subset_files = self.current_date_files[start_idx:end_idx]

        print("\n文件范围选择信息:")
        print(f"当前日期文件总数: {total_files}")
        print(f"选择的索引范围: [{start_idx}, {end_idx})")
        print(f"选择范围包含文件数: {len(subset_files)}")

        if subset_files:
            print(f"第一个文件: {subset_files[0]}")
            print(f"最后一个文件: {subset_files[-1]}")

        return subset_files

    @staticmethod
    def parse_persistence_diagram(filepath):
        """
        从txt文件中解析持续同调数据
        """
        dim0_pairs = []
        dim1_pairs = []

        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()

            pattern = re.compile(r'\((\d+),\s*\(([\d\.eE+\-infINF]+),\s*([\d\.eE+\-infINF]+)\)\)')
            matches = pattern.findall(content)

            for dim_str, birth_str, death_str in matches:
                try:
                    dim = int(dim_str)

                    if birth_str.lower() in {'inf', '+inf'}:
                        birth = np.inf
                    elif birth_str.lower() == '-inf':
                        birth = -np.inf
                    else:
                        birth = float(birth_str)

                    if death_str.lower() in {'inf', '+inf'}:
                        death = np.inf
                    elif death_str.lower() == '-inf':
                        death = -np.inf
                    else:
                        death = float(death_str)

                    if dim == 0:
                        dim0_pairs.append((birth, death))
                    elif dim == 1:
                        dim1_pairs.append((birth, death))
                except ValueError:
                    continue

            dim0_array = np.array(dim0_pairs) if dim0_pairs else np.empty((0, 2), dtype=float)
            dim1_array = np.array(dim1_pairs) if dim1_pairs else np.empty((0, 2), dtype=float)

            return {
                'dim0_pairs': dim0_array,
                'dim1_pairs': dim1_array
            }

        except Exception:
            return {
                'dim0_pairs': np.empty((0, 2), dtype=float),
                'dim1_pairs': np.empty((0, 2), dtype=float)
            }

    @staticmethod
    def compute_bottleneck_distance(pairs1, pairs2):
        """
        使用GUDHI计算瓶颈距离
        """
        try:
            if len(pairs1) == 0 or len(pairs2) == 0:
                return float('inf')

            distance = gd.bottleneck_distance(pairs1.tolist(), pairs2.tolist())
            return float(distance)

        except Exception:
            return float('inf')

    @staticmethod
    def process_comparison_chunk(target_dim0, target_dim1, work_items, distance_threshold):
        """
        在一个工作进程中处理一批比较任务（并行版本）
        
        Args:
            target_dim0: 目标的0维持续对
            target_dim1: 目标的1维持续对
            work_items: 一批其他日期文件的轻量信息
            distance_threshold: 距离阈值
            
        Returns:
            同时满足两个维度都小于阈值的点云列表
        """
        qualified_others = []

        for other_filename, other_path, other_date, other_stock_code in work_items:
            other_diagram = PersistenceDiagramProcessor.parse_persistence_diagram(other_path)

            # 计算0维和1维瓶颈距离
            dim0_distance = PersistenceDiagramProcessor.compute_bottleneck_distance(
                target_dim0, other_diagram['dim0_pairs']
            )
            dim1_distance = PersistenceDiagramProcessor.compute_bottleneck_distance(
                target_dim1, other_diagram['dim1_pairs']
            )

            # 检查是否同时满足两个维度都小于阈值
            if dim0_distance < distance_threshold and dim1_distance < distance_threshold:
                qualified_others.append({
                    'filename': other_filename,
                    'date': other_date,
                    'stock_code': other_stock_code,
                    'dim0_distance': dim0_distance,
                    'dim1_distance': dim1_distance,
                    'dim0_feature_count': len(other_diagram['dim0_pairs']),
                    'dim1_feature_count': len(other_diagram['dim1_pairs'])
                })

        return qualified_others

    def build_work_items(self):
        """
        构造其他日期文件的轻量任务列表
        """
        work_items = []

        for other_filename in self.other_date_files:
            info = self.file_info[other_filename]
            work_items.append(
                (
                    other_filename,
                    str(info['path']),
                    info.get('date', 'unknown'),
                    info.get('stock_code', 'unknown')
                )
            )

        return work_items

    def split_work_items_into_chunks(self, work_items):
        """
        将数万个文件比较任务切分成少量大批次
        """
        if not work_items:
            return []

        n_chunks = min(len(work_items), max(1, self.num_workers * 4))
        chunk_size = math.ceil(len(work_items) / n_chunks)

        return [work_items[i:i + chunk_size] for i in range(0, len(work_items), chunk_size)]

    def process_single_pointcloud_parallel(self, target_filename, min_count, distance_threshold,
                                            parallel_executor, comparison_chunks):
        """
        并行处理单个点云（新逻辑）
        
        Args:
            target_filename: 目标点云文件名
            min_count: 需要至少多少个点云同时满足条件
            distance_threshold: 距离阈值
            parallel_executor: 并行执行器
            comparison_chunks: 分割好的比较任务批次
            
        Returns:
            (是否满足条件, 结果字典)
        """
        # 检查目标文件是否存在
        if target_filename not in self.file_info:
            return False, None

        target_info = self.file_info[target_filename]

        # 解析目标点云的持续图
        target_diagram = self.parse_persistence_diagram(target_info['path'])
        target_dim0 = target_diagram['dim0_pairs']
        target_dim1 = target_diagram['dim1_pairs']

        if not comparison_chunks:
            return False, None

        # 并行处理所有比较任务
        chunk_results = parallel_executor(
            delayed(PersistenceDiagramProcessor.process_comparison_chunk)(
                target_dim0,
                target_dim1,
                chunk,
                distance_threshold
            )
            for chunk in comparison_chunks
        )

        # 合并所有结果
        qualified_others = []
        for chunk_qualified in chunk_results:
            qualified_others.extend(chunk_qualified)

        # 检查是否达到最小数量要求
        if len(qualified_others) >= min_count:
            # 按1维距离排序，取最小的min_count个
            qualified_others_sorted = sorted(qualified_others, key=lambda x: x['dim1_distance'])
            top_n_others = qualified_others_sorted[:min_count]

            # 准备结果
            top_n_pointclouds = []

            for i, item in enumerate(top_n_others, 1):
                top_n_pointclouds.append({
                    '排名': f"第{i}名",
                    '文件名': item['filename'],
                    '日期': item['date'],
                    '股票代码': item['stock_code'],
                    '0维距离': item['dim0_distance'],
                    '1维距离': item['dim1_distance'],
                    '0维特征数': item['dim0_feature_count'],
                    '1维特征数': item['dim1_feature_count']
                })

            # 统计信息
            all_dim0_distances = [item['dim0_distance'] for item in qualified_others]
            all_dim1_distances = [item['dim1_distance'] for item in qualified_others]

            results = {
                'target_info': {
                    'filename': target_filename,
                    'date': target_info['date'],
                    'stock_code': target_info['stock_code'],
                    'dim0_features': len(target_dim0),
                    'dim1_features': len(target_dim1),
                    'current_date': self.current_date
                },
                'top_n_pointclouds': top_n_pointclouds,
                'qualified_count': len(qualified_others),
                'min_count': min_count,
                'distance_threshold': distance_threshold,
                'total_files_processed': len(self.other_date_files),
                'all_dim0_distances': all_dim0_distances,
                'all_dim1_distances': all_dim1_distances
            }
            return True, results

        return False, None

    def process_selected_pointclouds_new(self, selected_files, min_count, distance_threshold, output_base_folder, range_info=""):
        """
        处理指定的点云列表，使用新的筛选逻辑（并行版本）
        
        Args:
            selected_files: 要处理的文件列表
            min_count: 需要至少多少个点云同时满足条件
            distance_threshold: 距离阈值
            output_base_folder: 输出基础文件夹路径
            range_info: 范围信息字符串，用于输出文件夹命名
            
        Returns:
            满足条件的点云列表
        """
        # 创建输出文件夹
        if range_info:
            output_folder = Path(output_base_folder) / f"{self.current_date}_筛选{range_info}"
        else:
            output_folder = Path(output_base_folder) / f"{self.current_date}_筛选"
        output_folder.mkdir(parents=True, exist_ok=True)

        print(f"\n输出文件夹: {output_folder}")
        print(f"将处理 {len(selected_files)} 个当前日期的点云")
        print(f"筛选条件: 至少 {min_count} 个第二类点云同时满足0维和1维距离都小于 {distance_threshold}")
        print("=" * 80)

        # 构建其他日期文件的比较任务
        work_items = self.build_work_items()
        comparison_chunks = self.split_work_items_into_chunks(work_items)

        if not comparison_chunks:
            print("错误: 没有可以进行比较的其他日期文件")
            return []

        print(f"其他日期文件数: {len(work_items):,}")
        print(f"并行任务批次数: {len(comparison_chunks)}")
        print(f"单个目标点云需要执行 {len(work_items):,} 次瓶颈距离计算")
        print("说明：外层进度条只有在一个目标点云的全部比较完成后才会增加 1。")

        # 存储所有满足条件的点云
        qualified_pointclouds = []

        # 进度条
        progress_bar = tqdm(selected_files, desc="处理点云", unit="个")

        # 使用loky后端，适合Windows
        with Parallel(n_jobs=self.num_workers, backend="loky", batch_size=1, pre_dispatch=self.num_workers) as parallel_executor:
            for filename in progress_bar:
                progress_bar.set_postfix_str(f"正在计算 {filename[:20]}")

                try:
                    is_qualified, results = self.process_single_pointcloud_parallel(
                        target_filename=filename,
                        min_count=min_count,
                        distance_threshold=distance_threshold,
                        parallel_executor=parallel_executor,
                        comparison_chunks=comparison_chunks
                    )

                except Exception as error:
                    print(f"\n处理文件 {filename} 时发生错误: {error}")
                    progress_bar.set_postfix_str(f"错误 {filename[:20]}")
                    continue

                if is_qualified and results is not None:
                    qualified_pointclouds.append(results)

                    # 保存结果到文件
                    self.save_results_new(results, output_folder)

                    progress_bar.set_postfix_str(
                        f"✓ {filename[:20]} (已找到 {len(qualified_pointclouds)} 个)"
                    )
                else:
                    progress_bar.set_postfix_str(f"✗ {filename[:20]}")

        print("\n" + "=" * 80)
        print("处理完成!")
        print(f"共找到 {len(qualified_pointclouds)} 个满足条件的点云")

        # 生成汇总报告
        if qualified_pointclouds:
            self.save_summary_report_new(qualified_pointclouds, output_folder, len(selected_files))

        return qualified_pointclouds

    def save_results_new(self, results, output_folder):
        """
        保存单个点云的结果到txt文件（新格式）
        """
        target_info = results['target_info']
        output_filename = f"top{results['min_count']}_similar_{target_info['date']}_{target_info['stock_code']}.txt"
        output_file = output_folder / output_filename

        with open(output_file, 'w', encoding='utf-8') as f:
            # 写入标题
            f.write("=" * 140 + "\n")
            f.write(f"点云相似性分析报告 (双阈值筛选)\n")
            f.write(f"目标点云: {target_info['filename']} (日期: {target_info['date']}, 股票: {target_info['stock_code']})\n")
            f.write(f"当前日期: {target_info['current_date']}\n")
            f.write("=" * 140 + "\n\n")

            # 目标点云信息
            f.write("【目标点云信息】\n")
            f.write("-" * 60 + "\n")
            f.write(f"文件名: {target_info['filename']}\n")
            f.write(f"日期: {target_info['date']}\n")
            f.write(f"股票代码: {target_info['stock_code']}\n")
            f.write(f"0维特征数量: {target_info['dim0_features']}\n")
            f.write(f"1维特征数量: {target_info['dim1_features']}\n")
            f.write(f"其他日期文件总数: {results['total_files_processed']}\n")
            f.write(f"距离阈值: {results['distance_threshold']}\n")
            f.write(f"最小匹配数要求: {results['min_count']}\n")
            f.write(f"实际找到的匹配数: {results['qualified_count']}\n")
            f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("\n")

            # 筛选条件说明
            f.write("【筛选条件说明】\n")
            f.write("-" * 60 + "\n")
            f.write(f"1. 第二类点云必须同时满足: 0维距离 < {results['distance_threshold']} 且 1维距离 < {results['distance_threshold']}\n")
            f.write(f"2. 至少需要 {results['min_count']} 个这样的点云\n")
            f.write(f"3. 按1维距离从小到大排序，取前 {results['min_count']} 个\n")
            f.write("\n")

            # 最相似的N个点云
            f.write(f"【按1维距离最小的 {results['min_count']} 个匹配点云】\n")
            f.write("=" * 140 + "\n")
            f.write("排名 | 文件名 | 日期 | 股票代码 | 0维距离 | 1维距离 | 0维特征数 | 1维特征数\n")
            f.write("-" * 140 + "\n")

            for idx, pointcloud in enumerate(results['top_n_pointclouds'], 1):
                f.write(f"{idx:3d} | {pointcloud['排名']:6s} | "
                       f"{pointcloud['文件名'][:40]:40s} | "
                       f"{pointcloud['日期']} | "
                       f"{pointcloud['股票代码']:10s} | "
                       f"{pointcloud['0维距离']:.6f} | "
                       f"{pointcloud['1维距离']:.6f} | "
                       f"{pointcloud['0维特征数']:8d} | "
                       f"{pointcloud['1维特征数']:8d}\n")

            f.write("=" * 140 + "\n\n")

            # 统计摘要
            f.write("【统计摘要】\n")
            f.write("-" * 80 + "\n")
            if results['all_dim0_distances']:
                f.write(f"所有匹配点云的0维距离统计:\n")
                f.write(f"  最小值: {min(results['all_dim0_distances']):.6f}\n")
                f.write(f"  最大值: {max(results['all_dim0_distances']):.6f}\n")
                f.write(f"  平均值: {np.mean(results['all_dim0_distances']):.6f}\n")
                f.write(f"  中位数: {np.median(results['all_dim0_distances']):.6f}\n")

            if results['all_dim1_distances']:
                f.write(f"\n所有匹配点云的1维距离统计:\n")
                f.write(f"  最小值: {min(results['all_dim1_distances']):.6f}\n")
                f.write(f"  最大值: {max(results['all_dim1_distances']):.6f}\n")
                f.write(f"  平均值: {np.mean(results['all_dim1_distances']):.6f}\n")
                f.write(f"  中位数: {np.median(results['all_dim1_distances']):.6f}\n")

            f.write("\n")
            f.write("=" * 140 + "\n")
            f.write("说明:\n")
            f.write(f"1. 共找到 {results['qualified_count']} 个同时满足双阈值条件的点云\n")
            f.write(f"2. 显示了其中1维距离最小的 {results['min_count']} 个\n")
            f.write("3. 距离值越小表示拓扑结构越相似\n")
            f.write("=" * 140 + "\n")

    def save_summary_report_new(self, qualified_pointclouds, output_folder, total_processed):
        """
        生成汇总报告（新格式）
        """
        summary_file = output_folder / f"summary_双阈值筛选_{self.current_date}.txt"

        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("=" * 120 + "\n")
            f.write("点云双阈值筛选汇总报告\n")
            f.write("=" * 120 + "\n\n")

            f.write(f"当前日期: {self.current_date}\n")
            f.write(f"筛选时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"处理的第一类文件数: {total_processed}\n")
            f.write(f"满足条件的点云数量: {len(qualified_pointclouds)}\n")
            f.write(f"筛选通过率: {len(qualified_pointclouds)/total_processed*100:.2f}%\n")
            f.write(f"第一类（当前日期）文件总数: {len(self.current_date_files)}\n")
            f.write(f"第二类（其他日期）文件总数: {len(self.other_date_files)}\n")

            if qualified_pointclouds:
                # 统计匹配数分布
                match_counts = [r['qualified_count'] for r in qualified_pointclouds]
                f.write(f"\n匹配数统计:\n")
                f.write(f"  最小匹配数: {min(match_counts)}\n")
                f.write(f"  最大匹配数: {max(match_counts)}\n")
                f.write(f"  平均匹配数: {np.mean(match_counts):.2f}\n")
                f.write(f"  中位匹配数: {np.median(match_counts)}\n")

            f.write("\n" + "=" * 120 + "\n")
            f.write("【满足条件的点云列表】\n")
            f.write("-" * 120 + "\n")
            f.write("序号 | 文件名 | 股票代码 | 0维特征数 | 1维特征数 | 匹配数\n")
            f.write("-" * 120 + "\n")

            for i, result in enumerate(qualified_pointclouds, 1):
                info = result['target_info']
                f.write(f"{i:3d} | {info['filename'][:40]:40s} | "
                       f"{info['stock_code']:10s} | "
                       f"{info['dim0_features']:8d} | "
                       f"{info['dim1_features']:8d} | "
                       f"{result['qualified_count']:6d}\n")

            f.write("\n")
            f.write("=" * 120 + "\n")
            f.write("详细报告请查看每个点云对应的txt文件\n")
            f.write("=" * 120 + "\n")

        print(f"汇总报告已保存: {summary_file}")


def main():
    """
    主函数
    """
    # 配置路径
    data_folder = r"D:\量化\我自己的实验\3rd test\persistence_results"
    output_base_folder = r"D:\量化\我自己的实验\3rd test\筛选结果"

    # 验证数据文件夹
    if not os.path.exists(data_folder):
        print(f"错误: 数据文件夹 '{data_folder}' 不存在")
        print("请修改 data_folder 变量为正确的路径")
        return

    # 检查GUDHI
    try:
        print(f"GUDHI版本: {gd.__version__}")
    except:
        print("请确保已安装GUDHI: pip install gudhi")
        return

    print("\n" + "=" * 80)
    print("点云双阈值筛选程序 v4.2 (并行版本 - 范围选择)")
    print("=" * 80)
    print("筛选逻辑:")
    print("1. 对每个第一类点云，计算与所有第二类点云的0维和1维Bottleneck距离")
    print("2. 记录同时满足: 0维距离 < 0.1 且 1维距离 < 0.1 的点云")
    print("3. 如果这样的点云数量 >= N，则按1维距离排序，取前N个")
    print("=" * 80)

    # 用户输入当前日期
    current_date = input("\n请输入当前日期 (格式: YYYYMMDD，例如: 20240628): ").strip()

    if len(current_date) != 8 or not current_date.isdigit():
        print("错误: 日期格式不正确，请使用8位数字格式")
        return

    # 用户输入最小匹配数
    try:
        min_count = int(input("请输入最小匹配数 N (例如: 5，表示至少需要5个点云同时满足条件): ").strip())
        if min_count <= 0:
            print("错误: N必须为正整数")
            return
    except ValueError:
        print("错误: 请输入有效的整数")
        return

    # 用户输入起始和结束索引
    try:
        start_idx = int(input("请输入起始索引 M（计数从0开始，包含）: ").strip())
        if start_idx < 0:
            print("错误: 起始索引不能为负数")
            return
    except ValueError:
        print("错误: 请输入有效的整数")
        return

    try:
        end_idx = int(input("请输入结束索引 N（计数从0开始，不包含）: ").strip())
        if end_idx <= start_idx:
            print("错误: 结束索引必须大于起始索引")
            return
    except ValueError:
        print("错误: 请输入有效的整数")
        return

    # 距离阈值固定为0.1
    distance_threshold = 0.1
    print(f"距离阈值: {distance_threshold} (固定值)")

    # 并行进程数设置
    max_workers = max(1, cpu_count() - 1)
    try:
        num_workers_input = input(f"请输入并行进程数 (默认: {max_workers}，回车使用默认): ").strip()
        if num_workers_input:
            num_workers = int(num_workers_input)
            if num_workers <= 0 or num_workers > cpu_count():
                print(f"进程数超出范围，使用默认值 {max_workers}")
                num_workers = max_workers
        else:
            num_workers = max_workers
    except ValueError:
        num_workers = max_workers
        print(f"输入无效，使用默认进程数: {num_workers}")

    # 初始化处理器
    print("\n" + "=" * 80)
    processor = PersistenceDiagramProcessor(data_folder, current_date, num_workers=num_workers)

    # 检查是否有当前日期的文件
    if len(processor.current_date_files) == 0:
        print(f"错误: 未找到日期为 {current_date} 的文件")
        print("请检查日期是否正确")
        return

    # 获取指定范围的文件
    selected_files = processor.get_subset_files_by_range(start_idx, end_idx)

    if len(selected_files) == 0:
        print(f"错误: 索引范围 [{start_idx}, {end_idx}) 没有文件")
        return

    # 显示配置信息
    print("\n" + "=" * 80)
    print("筛选配置:")
    print(f"当前日期: {current_date}")
    print(f"第一类文件总数: {len(processor.current_date_files)}")
    print(f"第二类文件总数: {len(processor.other_date_files)}")
    print(f"文件索引范围: [{start_idx}, {end_idx})")
    print(f"本范围文件数: {len(selected_files)}")
    print(f"最小匹配数 N: {min_count}")
    print(f"距离阈值: {distance_threshold}")
    print(f"筛选条件: 同时满足 0维<{distance_threshold} 且 1维<{distance_threshold}")
    print(f"并行进程数: {processor.num_workers}")
    
    # 创建范围信息字符串用于输出文件夹命名
    range_info = f"_范围{start_idx}_{end_idx}"
    output_folder_path = os.path.join(output_base_folder, f"{current_date}_筛选{range_info}")
    print(f"输出文件夹: {output_folder_path}")
    print("=" * 80)

    # 确认执行
    response = input("\n是否开始筛选？(y/n): ")
    if response.lower() != 'y':
        print("程序已取消")
        return

    # 开始处理
    print("\n开始筛选...")
    start_time = time.time()

    qualified_results = processor.process_selected_pointclouds_new(
        selected_files=selected_files,
        min_count=min_count,
        distance_threshold=distance_threshold,
        output_base_folder=output_base_folder,
        range_info=range_info
    )

    # 计算总耗时
    total_time = time.time() - start_time

    # 显示最终结果
    print("\n" + "=" * 80)
    print("筛选完成!")
    print("=" * 80)
    print(f"当前日期: {current_date}")
    print(f"索引范围 [{start_idx}, {end_idx}) 处理完成")
    print(f"本范围处理文件数: {len(selected_files)}")
    print(f"本范围满足条件的文件数: {len(qualified_results)}")
    if len(selected_files) > 0:
        print(f"本范围筛选通过率: {len(qualified_results)/len(selected_files)*100:.2f}%")
    print(f"输出文件夹: {output_folder_path}")
    print(f"总耗时: {total_time:.2f}秒 ({total_time/60:.2f}分钟)")
    print(f"使用并行进程数: {processor.num_workers}")

    if qualified_results:
        print(f"\n本范围满足条件的点云列表:")
        for i, result in enumerate(qualified_results[:10], 1):
            info = result['target_info']
            print(f"{i:2d}. {info['filename']} (股票: {info['stock_code']}, 匹配数: {result['qualified_count']})")
        if len(qualified_results) > 10:
            print(f"... 还有 {len(qualified_results) - 10} 个点云")

    print("=" * 80)


if __name__ == "__main__":
    main()

第10步，调取未来5天数据：上一步的代码从2024年6月28日的全部4977个点云切片中找出了530个值得研究的点云切片，此处要求N=5，以下代码实现的是把这530个点云切片及其top5 相似点云切片的后5天的交易数据调取出来，生成对应的csv文件，共530个，命名还是以这530个点云切片来命名。

In [ ]:
from __future__ import annotations

import csv
import io
import re
import shutil
import sys
import tempfile
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Iterable, Sequence, List, Dict, Set, Tuple, Optional

import pandas as pd
from tqdm import tqdm

# 如果是在 Jupyter 中，可以显示进度条
try:
    from IPython.display import display, HTML
    IN_IPYTHON = True
except ImportError:
    IN_IPYTHON = False


# 正则表达式模式
TARGET_FILE_RE = re.compile(
    r"^top5_similar_(?P<date>20\d{6})_(?P<code>\d{6})(?P<market>SZ|SH)\.txt$",
    re.IGNORECASE,
)
POINT_CLOUD_RE = re.compile(
    r"(?P<date>20\d{6})[_-]?(?P<code>\d{6})(?P<market>SZ|SH)\.txt",
    re.IGNORECASE,
)
STOCK_FILE_RE = re.compile(
    r"^(?P<code>\d{6})[._-]?(?P<market>SZ|SH)\.csv$",
    re.IGNORECASE,
)


@dataclass(frozen=True)
class PointCloud:
    """点云数据结构"""
    cloud_date: date
    stock_code: str

    @property
    def compact_date(self) -> str:
        return self.cloud_date.strftime("%Y%m%d")

    @property
    def compact_code(self) -> str:
        return self.stock_code.replace(".", "")

    @property
    def output_stem(self) -> str:
        return f"{self.compact_date}_{self.compact_code}"


@dataclass(frozen=True)
class SelectionGroup:
    """筛选结果组"""
    target: PointCloud
    similar: tuple[PointCloud, ...]
    source_file: Path


def normalize_stock_code(code: str, market: str) -> str:
    """标准化股票代码"""
    return f"{code}.{market.upper()}"


def point_cloud_from_match(match: re.Match[str]) -> PointCloud:
    """从正则匹配创建点云对象"""
    return PointCloud(
        cloud_date=datetime.strptime(match.group("date"), "%Y%m%d").date(),
        stock_code=normalize_stock_code(match.group("code"), match.group("market")),
    )


def read_text_flexibly(path: Path) -> str:
    """灵活读取文本文件（支持多种编码）"""
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-8", "gb18030"):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="ignore")


def find_selection_files(selection_dir: Path) -> List[Path]:
    """查找所有筛选结果文件"""
    files = sorted(
        path
        for path in selection_dir.rglob("*.txt")  # 修改为读取所有txt文件
        if TARGET_FILE_RE.fullmatch(path.name)  # 仍然只处理符合命名规则的文件
    )
    if not files:
        # 如果没有符合命名规则的文件，尝试读取所有txt文件
        files = sorted(
            path
            for path in selection_dir.rglob("*.txt")
            if "top5_similar" in path.name.lower()
        )
        if not files:
            raise ValueError(f"在 {selection_dir} 中没有找到任何 top5_similar_*.txt 文件")
        else:
            print(f"找到 {len(files)} 个包含 top5_similar 的txt文件（不符合标准命名规则）")
    else:
        print(f"找到 {len(files)} 个筛选结果文件")
    return files


def parse_selection_file(path: Path) -> SelectionGroup:
    """解析筛选结果文件"""
    target_match = TARGET_FILE_RE.fullmatch(path.name)
    if target_match is None:
        # 如果不是标准命名格式，尝试从文件内容中提取目标点云信息
        content = read_text_flexibly(path)
        # 尝试从内容中找第一个点云作为目标
        matches = list(POINT_CLOUD_RE.finditer(content))
        if not matches:
            raise ValueError(f"无法从文件 {path.name} 中解析点云信息")
        
        # 使用第一个匹配作为目标点云
        target = point_cloud_from_match(matches[0])
        print(f"  从文件内容中提取目标点云: {target.compact_date}_{target.compact_code}")
        
        similar: List[PointCloud] = []
        seen: Set[PointCloud] = set()
        seen.add(target)  # 先将目标加入已见集合
        
        for match in matches[1:]:  # 从第二个开始作为相似点云
            point_cloud = point_cloud_from_match(match)
            if point_cloud in seen:
                continue
            seen.add(point_cloud)
            similar.append(point_cloud)
        
        if len(similar) != 5:
            print(f"警告: {path.name} 包含 {len(similar)} 个相似点云（期望5个）")
        
        return SelectionGroup(target=target, similar=tuple(similar[:5]), source_file=path)
    
    # 标准命名格式的处理
    target = point_cloud_from_match(target_match)
    similar: List[PointCloud] = []
    seen: Set[PointCloud] = set()
    content = read_text_flexibly(path)
    
    for match in POINT_CLOUD_RE.finditer(content):
        point_cloud = point_cloud_from_match(match)
        if point_cloud == target or point_cloud in seen:
            continue
        seen.add(point_cloud)
        similar.append(point_cloud)

    if len(similar) != 5:
        print(f"警告: {path.name} 包含 {len(similar)} 个相似点云（期望5个）")
    
    return SelectionGroup(target=target, similar=tuple(similar[:5]), source_file=path)


def canonical_code_from_stock_filename(filename: str) -> Optional[str]:
    """从股票文件名提取标准化代码"""
    match = STOCK_FILE_RE.fullmatch(Path(filename).name)
    if match is None:
        return None
    return normalize_stock_code(match.group("code"), match.group("market"))


def index_stock_directory(stock_dir: Path) -> Dict[str, Path]:
    """索引股票CSV文件"""
    index: Dict[str, Path] = {}
    for path in stock_dir.rglob("*.csv"):
        stock_code = canonical_code_from_stock_filename(path.name)
        if stock_code is None:
            continue
        if stock_code in index:
            raise ValueError(
                f"股票 {stock_code} 对应多个 CSV：{index[stock_code]}、{path}"
            )
        index[stock_code] = path
    if not index:
        raise ValueError(f"在 {stock_dir} 中没有找到可识别的股票 CSV")
    return index


def parse_source_date(value: str) -> date:
    """解析日期字符串"""
    text = value.strip()
    if not text:
        raise ValueError("日期为空")

    # 尝试 ISO 格式
    iso_candidate = text.split("T", 1)[0].split(" ", 1)[0]
    try:
        return date.fromisoformat(iso_candidate.replace("/", "-"))
    except ValueError:
        pass

    # 尝试其他常见格式
    for date_format in (
        "%Y%m%d",
        "%Y/%m/%d",
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%m/%d/%y",
    ):
        try:
            return datetime.strptime(text, date_format).date()
        except ValueError:
            continue
    raise ValueError(f"不支持的日期格式：{value!r}")


def decode_csv(path: Path) -> Tuple[str, str]:
    """解码CSV文件"""
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-8", "gb18030"):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"无法识别 CSV 编码：{path}")


def load_next_five_trading_days(
    csv_path: Path, cloud_date: date
) -> Tuple[List[str], List[List[str]], Tuple[date, ...]]:
    """加载股票在指定日期后的5个交易日数据"""
    text, _ = decode_csv(csv_path)
    reader = csv.reader(io.StringIO(text))
    
    try:
        header = next(reader)
    except StopIteration as exc:
        raise ValueError(f"CSV 是空文件：{csv_path}") from exc

    # 查找日期列
    normalized_headers = {
        column.strip().lower().replace("_", ""): index
        for index, column in enumerate(header)
    }
    date_index = normalized_headers.get("eventdate")
    if date_index is None:
        date_index = normalized_headers.get("date")
    if date_index is None:
        raise ValueError(f"{csv_path.name} 缺少 EventDate/Date 日期列")

    # 收集日期数据
    dated_rows: List[Tuple[date, int, List[str]]] = []
    for row_number, row in enumerate(reader, start=2):
        if not row or all(not cell.strip() for cell in row):
            continue
        if len(row) != len(header):
            print(f"警告: {csv_path.name} 第 {row_number} 行列数不匹配，跳过")
            continue
        try:
            row_date = parse_source_date(row[date_index])
        except ValueError:
            continue
        if row_date > cloud_date:
            dated_rows.append((row_date, row_number, row))

    # 获取前5个交易日
    trading_dates = sorted({row_date for row_date, _, _ in dated_rows})[:5]
    if len(trading_dates) < 5:
        print(f"警告: {csv_path.name} 在 {cloud_date.isoformat()} 之后只有 {len(trading_dates)} 个交易日")
        if not trading_dates:
            raise ValueError(f"{csv_path.name} 在 {cloud_date.isoformat()} 之后没有交易日数据")

    selected_dates = set(trading_dates)
    selected_rows = [
        row
        for row_date, _, row in sorted(dated_rows, key=lambda item: (item[0], item[1]))
        if row_date in selected_dates
    ]
    return header, selected_rows, tuple(trading_dates)


def padded_label(label: str, column_count: int) -> List[str]:
    """创建带标签的填充行"""
    return [label, *([""] * (column_count - 1))]


def blank_row(column_count: int) -> List[str]:
    """创建空白行"""
    return [""] * column_count


def collect_group_rows(
    group: SelectionGroup, stock_index: Dict[str, Path]
) -> Tuple[List[str], List[List[str]], List[str]]:
    """收集一个组的行数据"""
    clouds = (group.target, *group.similar)
    common_header: List[str] | None = None
    blocks: List[List[List[str]]] = []
    audit_lines: List[str] = []

    for cloud in clouds:
        if cloud.stock_code not in stock_index:
            print(f"警告: 股票 {cloud.stock_code} 在股票数据中不存在，跳过")
            continue
            
        header, rows, trading_dates = load_next_five_trading_days(
            stock_index[cloud.stock_code], cloud.cloud_date
        )
        if common_header is None:
            common_header = header
        elif header != common_header:
            print(f"警告: {stock_index[cloud.stock_code].name} 的列名与其他股票不一致，尝试对齐")
            common_cols = set(common_header) & set(header)
            if common_cols:
                common_header = sorted(common_cols)
            else:
                raise ValueError(f"股票列名完全不兼容: {common_header} vs {header}")
        
        blocks.append(rows)
        audit_lines.append(
            f"{cloud.compact_date}_{cloud.compact_code}: "
            + ", ".join(item.isoformat() for item in trading_dates)
        )

    if not common_header:
        raise ValueError(f"无法获取任何股票数据：{group.target.stock_code}")

    # 构建输出行
    output_rows: List[List[str]] = [
        common_header,
        padded_label("目标点云", len(common_header)),
        *blocks[0],
        blank_row(len(common_header)),
        padded_label("相似点云", len(common_header)),
    ]
    for index, rows in enumerate(blocks[1:]):
        if index > 0:
            output_rows.append(blank_row(len(common_header)))
        output_rows.extend(rows)
    return common_header, output_rows, audit_lines


def write_output_csv(path: Path, rows: Iterable[Sequence[str]]) -> None:
    """写入CSV文件"""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerows(rows)


def collect_required_codes(groups: Iterable[SelectionGroup]) -> Set[str]:
    """收集所有需要的股票代码"""
    return {
        cloud.stock_code
        for group in groups
        for cloud in (group.target, *group.similar)
    }


def run_notebook(selection_folder: Path, stock_folder: Path, output_folder: Path) -> List[Path]:
    """
    在 Jupyter Notebook 中运行的主函数
    
    Args:
        selection_folder: 筛选结果文件夹路径
        stock_folder: 股票CSV文件夹路径
        output_folder: 输出文件夹路径
        
    Returns:
        生成的文件路径列表
    """
    selection_folder = Path(selection_folder).resolve()
    stock_folder = Path(stock_folder).resolve()
    output_folder = Path(output_folder).resolve()

    print("=" * 80)
    print("点云数据收集工具 (Jupyter版)")
    print("=" * 80)
    print(f"筛选结果文件夹: {selection_folder}")
    print(f"股票数据文件夹: {stock_folder}")
    print(f"输出文件夹: {output_folder}")
    print("=" * 80)

    # 检查文件夹是否存在
    if not selection_folder.exists():
        raise FileNotFoundError(f"筛选结果文件夹不存在: {selection_folder}")
    if not stock_folder.exists():
        raise FileNotFoundError(f"股票数据文件夹不存在: {stock_folder}")

    # 查找筛选结果文件
    print("\n正在查找筛选结果文件...")
    selection_files = find_selection_files(selection_folder)
    
    if not selection_files:
        raise ValueError("没有找到任何筛选结果文件")
    
    print(f"找到 {len(selection_files)} 个筛选结果文件")

    # 解析筛选结果
    print("\n正在解析筛选结果...")
    groups: List[SelectionGroup] = []
    for path in tqdm(selection_files, desc="解析文件", unit="个"):
        try:
            group = parse_selection_file(path)
            groups.append(group)
        except Exception as e:
            print(f"警告: 解析 {path.name} 时出错: {e}")
            continue

    print(f"成功解析 {len(groups)} 个筛选结果")

    if not groups:
        raise ValueError("没有成功解析任何筛选结果文件")

    # 收集需要的股票代码
    required_codes = collect_required_codes(groups)
    print(f"需要 {len(required_codes)} 个股票的数据")

    # 索引股票数据
    print("\n正在索引股票数据...")
    stock_index = index_stock_directory(stock_folder)
    print(f"找到 {len(stock_index)} 个股票CSV文件")

    # 检查缺失的股票
    missing_codes = required_codes - set(stock_index.keys())
    if missing_codes:
        print(f"警告: 以下 {len(missing_codes)} 个股票数据缺失:")
        for code in sorted(missing_codes)[:10]:
            print(f"  - {code}")
        if len(missing_codes) > 10:
            print(f"  ... 还有 {len(missing_codes) - 10} 个")

    # 创建输出文件夹
    output_folder.mkdir(parents=True, exist_ok=True)

    # 处理每个组
    print("\n正在收集数据...")
    generated: List[Path] = []
    error_count = 0

    for group in tqdm(groups, desc="处理点云组", unit="组"):
        try:
            _, rows, audit_lines = collect_group_rows(group, stock_index)
            output_path = output_folder / f"{group.target.output_stem}.csv"
            write_output_csv(output_path, rows)
            generated.append(output_path)
            print(f"  ✓ {output_path.name}")
            for line in audit_lines:
                print(f"       {line}")
        except Exception as e:
            error_count += 1
            print(f"  ✗ 处理 {group.target.output_stem} 时出错: {e}")

    print("\n" + "=" * 80)
    print("处理完成!")
    print(f"成功生成: {len(generated)} 个CSV文件")
    if error_count > 0:
        print(f"失败: {error_count} 个")
    print(f"输出文件夹: {output_folder}")
    print("=" * 80)

    return generated


# ========== Jupyter Notebook 使用示例 ==========

def main_notebook():
    """
    Jupyter Notebook 主函数
    """
    # 使用您指定的文件夹路径
    selection_folder = r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select"
    stock_folder = r"D:\量化\我自己的实验\3rd test\stock"  # 请修改为您的股票数据文件夹
    # 在同一个文件夹下创建新的输出文件夹
    output_folder = r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果"

    try:
        generated = run_notebook(selection_folder, stock_folder, output_folder)
        
        # 显示生成的文件列表
        print(f"\n生成的文件列表 (共 {len(generated)} 个):")
        for i, path in enumerate(generated[:20], 1):
            print(f"  {i:3d}. {path.name}")
        if len(generated) > 20:
            print(f"  ... 还有 {len(generated) - 20} 个文件")
            
    except Exception as e:
        print(f"执行失败: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    # 如果在 Jupyter 中运行，使用 notebook 版本
    if IN_IPYTHON or 'get_ipython' in globals():
        main_notebook()
    else:
        # 如果在命令行中运行
        def main_cli():
            print("正在处理点云数据...")
            main_notebook()
        main_cli()

第11步，计算涨跌：在这个文件夹中，D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果，读取所有csv，每个csv中有一个目标点云和5个相似点云各5天的交易日线数据，读取6个点云的第一个交易日的prev_close数据c0和close数据c1, 还有第k个交易日的close数据ck(k<=5), 生成新的csv，把原csv中的SecuCode, EventDate, prev_close, close列抄过去，新增一列d1，其值为d1=c1-c0, 新增一列d1涨跌，若d1>=0, 则标记为1，表示涨，若d1<0则标记为0，表示跌。新增一列d2，其值为d2=c2-c0, 新增一列d2涨跌，若d2>=0则标记为1，否则标记为0。再新增3列d3, d4. d5以及3列d3涨跌，d4涨跌和d5涨跌，他们的值的计算参照类似d1和d2.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


def _load_full_stock(full_data_dir, secu_code):
    """按股票代码前缀定位完整历史CSV（如 000060SZ -> data/stock/000060.SZ.csv）。"""
    prefix = str(secu_code)[:6]
    matches = list(Path(full_data_dir).glob(f"{prefix}*.csv"))
    if not matches:
        return None
    try:
        return pd.read_csv(matches[0], encoding='utf-8-sig', low_memory=False)
    except Exception:
        return None


def _future_labels(full_df, cloud_date):
    """
    对齐 topoquant.load_future_directions：返回 {k:(diff, up)} for k=1..5。
    cloud_date = 窗口最后一天(W_end)；未来 = cloud_date 之后严格5个交易日；
    baseline = 未来第1日 prev_close；d_k = sign(close[未来第k日] - baseline)。
    若未来不足5日返回 None（与 topoquant 丢弃该目标一致）。
    """
    if full_df is None:
        return None
    frame = full_df.copy()
    frame["EventDate"] = pd.to_datetime(frame["EventDate"], errors="coerce")
    frame["prev_close"] = pd.to_numeric(frame["prev_close"], errors="coerce")
    frame["close"] = pd.to_numeric(frame["close"], errors="coerce")
    frame = frame.dropna(subset=["EventDate", "prev_close", "close"])
    frame = frame.loc[frame["EventDate"].dt.date > cloud_date]
    frame = frame.sort_values("EventDate", ascending=True, kind="stable")
    frame = frame.iloc[:5]
    if len(frame) != 5:
        return None
    baseline = float(frame.iloc[0]["prev_close"])
    out = {}
    for k in range(1, 6):
        close_k = float(frame["close"].iloc[k - 1])
        diff = round(close_k - baseline, 6)
        out[k] = (diff, 1 if diff >= 0 else 0)
    return out

def process_stock_data():
    """
    处理股票数据，计算每个点云【窗口之后真实未来5个交易日】的涨跌情况。

    ★ 标签口径已与 topoquant 对齐（样本外标签）★
    定义严格等价 src/topoquant/data.py : load_future_directions ：
      - 每个点云 = 一个 60 交易日窗口，窗口最后一天 = cloud_date(W_end)。
      - 未来第 k 日(k=1..5) = cloud_date 之后的第 k 个交易日。
      - baseline = 未来第 1 日的 prev_close。
      - d_k = 1 若 (close[未来第k日] - baseline) >= 0 否则 0。

    说明：
      - 持久图仍由窗口内 60 天构建（来自 处理结果/*.csv 的 60 行窗口）。
      - 标签需窗口之后的真实未来，故另从完整股票历史 full_data_dir 读取。
      - 标签写入每个点云的前 5 行（d1..d5 / d1涨跌..d5涨跌），
        predict_by_similar 与 analyze_prediction_accuracy 无需改动即可直接使用。
    """
    # 窗口CSV（持久图用，每云60行窗口）
    input_folder = Path("D:/量化/我自己的实验/3rd test/筛选结果/学生操作数据汇总/20240628_select/处理结果")
    # 完整股票历史（标签用，含窗口之后的未来交易日）。与 topoquant 同源：data/stock
    full_data_dir = Path("D:/TDA_IMP/8.3/8.3/data/stock")

    output_folder = input_folder / "processed_results"
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"输入文件夹: {input_folder}")
    print(f"完整历史目录: {full_data_dir}")
    print(f"输出文件夹: {output_folder}")
    print("=" * 80)

    csv_files = list(input_folder.glob("*.csv"))
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return

    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)

    processed_count = 0
    error_count = 0

    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')

            required_cols = ['SecuCode', 'EventDate', 'prev_close', 'close']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"\n警告: {csv_file.name} 缺少必要的列: {missing_cols}")
                error_count += 1
                continue

            grouped = df.groupby('SecuCode')
            processed_rows = []

            for secu_code, group in grouped:
                group_sorted = group.sort_values('EventDate').reset_index(drop=True)

                # 窗口最后一天 = cloud_date（W_end）
                cloud_date = pd.to_datetime(group_sorted['EventDate'].iloc[-1]).date()

                # 从完整历史读取该股票窗口之后的真实未来
                full = _load_full_stock(full_data_dir, secu_code)
                labels = _future_labels(full, cloud_date)

                for row_idx, row in group_sorted.iterrows():
                    new_row = row.to_dict()
                    for i in range(1, 6):
                        new_row[f'd{i}'] = None
                        new_row[f'd{i}涨跌'] = None
                    # 标签写入前5行（位置不变，predict/evaluate 无需改动）
                    if row_idx < 5 and labels is not None:
                        day_num = row_idx + 1
                        diff_v, up_v = labels[day_num]
                        new_row[f'd{day_num}'] = diff_v
                        new_row[f'd{day_num}涨跌'] = up_v
                    processed_rows.append(new_row)

            new_df = pd.DataFrame(processed_rows)
            original_cols = df.columns.tolist()
            new_cols = original_cols.copy()
            for i in range(1, 6):
                new_cols.append(f'd{i}')
                new_cols.append(f'd{i}涨跌')
            for col in new_cols:
                if col not in new_df.columns:
                    new_df[col] = None
            new_df = new_df[new_cols]

            output_file = output_folder / csv_file.name
            new_df.to_csv(output_file, encoding='utf-8-sig', index=False)
            processed_count += 1

        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            error_count += 1
            continue

    print("\n" + "=" * 80)
    print("处理完成!")
    print(f"成功处理: {processed_count} 个文件")
    if error_count > 0:
        print(f"失败: {error_count} 个文件")
    print(f"结果保存在: {output_folder}")
    print("=" * 80)

def process_stock_data_with_verification():
    """
    处理股票数据并显示验证信息（标签口径已与 topoquant 对齐：窗口之后真实未来）。
    """
    input_folder = Path("D:/量化/我自己的实验/3rd test/筛选结果/学生操作数据汇总/20240628_select/处理结果")
    full_data_dir = Path("D:/TDA_IMP/8.3/8.3/data/stock")
    output_folder = input_folder / "processed_results_verified"
    output_folder.mkdir(parents=True, exist_ok=True)

    csv_files = list(input_folder.glob("*.csv"))
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return

    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)

    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            required_cols = ['SecuCode', 'EventDate', 'prev_close', 'close']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"\n警告: {csv_file.name} 缺少必要的列: {missing_cols}")
                continue

            grouped = df.groupby('SecuCode')
            processed_rows = []

            for secu_code, group in grouped:
                group_sorted = group.sort_values('EventDate').reset_index(drop=True)
                cloud_date = pd.to_datetime(group_sorted['EventDate'].iloc[-1]).date()
                full = _load_full_stock(full_data_dir, secu_code)
                labels = _future_labels(full, cloud_date)

                print(f"\n文件: {csv_file.name}  股票: {secu_code}  cloud_date(W_end)={cloud_date}")
                if labels is None:
                    print("  未来不足5个交易日，跳过标签")
                else:
                    for k in range(1, 6):
                        diff_v, up_v = labels[k]
                        print(f"  未来第{k}日: diff={diff_v:.6f}, {'涨' if up_v else '跌'}")

                for row_idx, row in group_sorted.iterrows():
                    new_row = row.to_dict()
                    for i in range(1, 6):
                        new_row[f'd{i}'] = None
                        new_row[f'd{i}涨跌'] = None
                    if row_idx < 5 and labels is not None:
                        day_num = row_idx + 1
                        diff_v, up_v = labels[day_num]
                        new_row[f'd{day_num}'] = diff_v
                        new_row[f'd{day_num}涨跌'] = up_v
                    processed_rows.append(new_row)

            new_df = pd.DataFrame(processed_rows)
            original_cols = df.columns.tolist()
            new_cols = original_cols.copy()
            for i in range(1, 6):
                new_cols.append(f'd{i}')
                new_cols.append(f'd{i}涨跌')
            for col in new_cols:
                if col not in new_df.columns:
                    new_df[col] = None
            new_df = new_df[new_cols]
            output_file = output_folder / csv_file.name
            new_df.to_csv(output_file, encoding='utf-8-sig', index=False)

        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            continue



第12步，做出预测：我有一个文件夹，D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results，请你阅读每一个csv，里面有5列dk涨跌，k取1到5. 每个csv共包含6个点云，其中第一个是目标点云，共5行数据，即5个交易日。剩下的5个是相似点云也是每个点云有5行数据。接下来帮我写一个简单的判断逻辑来预测目标点云的5日涨跌趋势：新增一列，名为预测值，若5个相似点云的d1涨跌列的数值中有3个或以上是1，则判断目标点云也是涨，在目标点云的第一行和预测值列相交的格子里填1，若5个相似点云的d1涨跌列的数值中有3个或以上是0，则判断目标点云是跌，在那个格子里填0。若5个相似点云的d2涨跌列的数值中有3个或以上是1，则在目标点云的第2行和预测值列相交的格子里填1，后面依此类推。所有的操作都在源文件中执行。


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def predict_by_similar():
    """
    根据相似点云的涨跌情况预测目标点云每个交易日的涨跌
    - d1涨跌: 5个相似点云中如果有3个或以上是1，则预测目标为涨(1)，否则为跌(0)
    - 预测值写入目标点云的第1行（对应d1）
    - d2涨跌到d5涨跌同理，分别写入第2行到第5行
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    # 查找所有CSV文件
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 处理每个CSV文件
    processed_count = 0
    error_count = 0
    
    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            # 读取CSV文件
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            # 检查必要的列是否存在
            required_cols = ['SecuCode', 'd1涨跌', 'd2涨跌', 'd3涨跌', 'd4涨跌', 'd5涨跌']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"\n警告: {csv_file.name} 缺少必要的列: {missing_cols}")
                error_count += 1
                continue
            
            # 添加预测值列（如果不存在）
            if '预测值' not in df.columns:
                df['预测值'] = None
            
            # 按股票代码分组
            grouped = df.groupby('SecuCode')
            
            # 获取所有股票代码
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) < 6:
                print(f"\n警告: {csv_file.name} 中股票代码数量不足6个")
                error_count += 1
                continue
            
            target_code = secu_codes[0]  # 第一个是目标点云
            similar_codes = secu_codes[1:6]  # 剩下5个是相似点云
            
            print(f"\n处理文件: {csv_file.name}")
            print(f"  目标点云: {target_code}")
            print(f"  相似点云: {similar_codes}")
            
            # 收集相似点云每个交易日的涨跌数据
            # 每个相似点云有5行数据，分别对应5个交易日
            similar_data = []
            for code in similar_codes:
                code_data = df[df['SecuCode'] == code]
                if len(code_data) >= 5:
                    # 获取该点云5天的涨跌数据
                    d_values = []
                    for day in range(1, 6):
                        col_name = f'd{day}涨跌'
                        # 取第day-1行的数据（因为索引从0开始）
                        if day-1 < len(code_data):
                            d_values.append(code_data.iloc[day-1][col_name])
                        else:
                            d_values.append(None)
                    similar_data.append({
                        'code': code,
                        'd1': d_values[0],
                        'd2': d_values[1],
                        'd3': d_values[2],
                        'd4': d_values[3],
                        'd5': d_values[4]
                    })
            
            # 对每个交易日进行预测
            predictions = {}
            for day in range(1, 6):
                col_name = f'd{day}涨跌'
                
                # 统计相似点云中该交易日涨跌为1的个数
                count_1 = sum(1 for data in similar_data if data[f'd{day}'] == 1)
                count_0 = sum(1 for data in similar_data if data[f'd{day}'] == 0)
                
                # 如果有3个或以上是1，预测为涨(1)，否则预测为跌(0)
                if count_1 >= 3:
                    prediction = 1
                else:
                    prediction = 0
                
                predictions[day] = prediction
                
                print(f"    d{day}涨跌: 相似点云中1的数量={count_1}, 0的数量={count_0}, 预测值={prediction}")
            
            # 将预测值写入目标点云对应的行
            target_rows = df[df['SecuCode'] == target_code]
            if len(target_rows) >= 5:
                # 获取目标点云的行索引
                target_indices = target_rows.index.tolist()
                
                # 对每个交易日，写入对应的行
                for day in range(1, 6):
                    row_idx = target_indices[day - 1]  # 第1行对应d1，第2行对应d2，依此类推
                    prediction = predictions[day]
                    df.at[row_idx, '预测值'] = prediction
                
                print(f"  预测值已写入目标点云:")
                for day in range(1, 6):
                    print(f"    第{day}行: d{day}预测值={predictions[day]}")
            else:
                print(f"  警告: 目标点云 {target_code} 的数据不足5行")
            
            # 保存修改后的文件
            df.to_csv(csv_file, encoding='utf-8-sig', index=False)
            processed_count += 1
            
        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            error_count += 1
            continue
    
    print("\n" + "=" * 80)
    print("处理完成!")
    print(f"成功处理: {processed_count} 个文件")
    if error_count > 0:
        print(f"失败: {error_count} 个文件")
    print("=" * 80)

def predict_by_similar_with_summary():
    """
    根据相似点云的涨跌情况预测目标点云的涨跌（带汇总信息）
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 存储所有预测统计
    all_predictions = []
    
    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            required_cols = ['SecuCode', 'd1涨跌', 'd2涨跌', 'd3涨跌', 'd4涨跌', 'd5涨跌']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"\n警告: {csv_file.name} 缺少必要的列: {missing_cols}")
                continue
            
            if '预测值' not in df.columns:
                df['预测值'] = None
            
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) < 6:
                continue
            
            target_code = secu_codes[0]
            similar_codes = secu_codes[1:6]
            
            # 收集相似点云的涨跌数据
            similar_data = []
            for code in similar_codes:
                code_data = df[df['SecuCode'] == code]
                if len(code_data) >= 5:
                    d_values = []
                    for day in range(1, 6):
                        col_name = f'd{day}涨跌'
                        if day-1 < len(code_data):
                            d_values.append(code_data.iloc[day-1][col_name])
                        else:
                            d_values.append(None)
                    similar_data.append({
                        'code': code,
                        'd1': d_values[0],
                        'd2': d_values[1],
                        'd3': d_values[2],
                        'd4': d_values[3],
                        'd5': d_values[4]
                    })
            
            # 计算预测值
            predictions = {}
            for day in range(1, 6):
                col_name = f'd{day}涨跌'
                count_1 = sum(1 for data in similar_data if data[f'd{day}'] == 1)
                count_0 = sum(1 for data in similar_data if data[f'd{day}'] == 0)
                
                prediction = 1 if count_1 >= 3 else 0
                predictions[day] = prediction
            
            # 写入预测值
            target_rows = df[df['SecuCode'] == target_code]
            if len(target_rows) >= 5:
                target_indices = target_rows.index.tolist()
                for day in range(1, 6):
                    row_idx = target_indices[day - 1]
                    df.at[row_idx, '预测值'] = predictions[day]
            
            # 保存文件
            df.to_csv(csv_file, encoding='utf-8-sig', index=False)
            
            # 记录预测统计
            all_predictions.append({
                'file': csv_file.name,
                'target': target_code,
                'd1': predictions[1],
                'd2': predictions[2],
                'd3': predictions[3],
                'd4': predictions[4],
                'd5': predictions[5]
            })
            
        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            continue
    
    # 生成汇总报告
    print("\n" + "=" * 80)
    print("预测汇总报告")
    print("=" * 80)
    
    if all_predictions:
        # 创建汇总DataFrame
        summary_df = pd.DataFrame(all_predictions)
        
        # 显示预测分布
        print("\n预测值分布:")
        for day in ['d1', 'd2', 'd3', 'd4', 'd5']:
            count_1 = (summary_df[day] == 1).sum()
            count_0 = (summary_df[day] == 0).sum()
            print(f"  {day}: 预测涨({1})={count_1}个, 预测跌({0})={count_0}个")
        
        # 显示详细预测结果（前10个）
        print("\n详细预测结果（前10个文件）:")
        display_cols = ['file', 'target', 'd1', 'd2', 'd3', 'd4', 'd5']
        print(summary_df[display_cols].head(10).to_string(index=False))
        
        # 保存汇总报告
        summary_file = input_folder / "预测汇总报告.csv"
        summary_df.to_csv(summary_file, encoding='utf-8-sig', index=False)
        print(f"\n汇总报告已保存: {summary_file}")
    
    print("\n" + "=" * 80)
    print("处理完成!")
    print("=" * 80)

def check_predictions():
    """
    检查已处理的文件中的预测值
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    for csv_file in csv_files[:5]:  # 只检查前5个
        print(f"\n文件: {csv_file.name}")
        
        df = pd.read_csv(csv_file, encoding='utf-8-sig')
        
        if '预测值' in df.columns:
            # 显示目标点云的预测值
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) > 0:
                target_code = secu_codes[0]
                target_rows = df[df['SecuCode'] == target_code]
                
                if len(target_rows) >= 5:
                    print(f"  目标点云: {target_code}")
                    print(f"  预测值:")
                    for idx, row in target_rows.iterrows():
                        prediction = row.get('预测值', None)
                        day = target_rows.index.get_loc(idx) + 1
                        print(f"    第{day}行 (d{day}): {prediction}")
        else:
            print("  没有预测值列")

def analyze_prediction_accuracy():
    """
    分析预测准确率（如果有实际值的话）
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 存储准确率统计
    accuracy_stats = {day: {'correct': 0, 'total': 0} for day in range(1, 6)}
    
    for csv_file in tqdm(csv_files, desc="分析文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            if '预测值' not in df.columns:
                continue
            
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) > 0:
                target_code = secu_codes[0]
                target_rows = df[df['SecuCode'] == target_code]
                
                if len(target_rows) >= 5:
                    # 对每个交易日检查预测值
                    for day in range(1, 6):
                        col_name = f'd{day}涨跌'
                        if col_name in df.columns:
                            # 获取目标点云该日的实际涨跌值
                            actual_value = target_rows.iloc[day-1][col_name]
                            predicted_value = target_rows.iloc[day-1]['预测值']
                            
                            if pd.notna(predicted_value):
                                accuracy_stats[day]['total'] += 1
                                if actual_value == predicted_value:
                                    accuracy_stats[day]['correct'] += 1
                                
        except Exception as e:
            print(f"\n分析 {csv_file.name} 时出错: {e}")
            continue
    
    # 显示准确率
    print("\n" + "=" * 80)
    print("预测准确率分析")
    print("=" * 80)
    
    for day in range(1, 6):
        total = accuracy_stats[day]['total']
        correct = accuracy_stats[day]['correct']
        if total > 0:
            accuracy = correct / total * 100
            print(f"d{day}: 正确={correct}, 总预测={total}, 准确率={accuracy:.2f}%")
        else:
            print(f"d{day}: 没有预测数据")

# ========== 在 Jupyter Notebook 中使用 ==========

def main():
    """主函数 - 标准预测"""
    predict_by_similar()

def main_with_summary():
    """带汇总信息的预测"""
    predict_by_similar_with_summary()

def main_check():
    """检查预测结果"""
    check_predictions()

def main_analyze():
    """分析预测准确率"""
    analyze_prediction_accuracy()

# 在 Jupyter 中执行
if __name__ == "__main__":
    # 标准预测
    main()
    
    # 如果需要带汇总信息，取消注释下面的行
    # main_with_summary()
    # main_check()
    # main_analyze()

第13步，统计预测结果：

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def evaluate_predictions():
    """
    比较目标点云的实际涨跌和预测值
    - 如果目标点云的d1涨跌与预测值同为1或同为0，则预测结果记为1（正确）
    - 如果目标点云的d1涨跌与预测值一个是1一个是0，则预测结果记为0（错误）
    - d2到d5同理
    - 预测结果写入'预测结果'列，对应目标点云的每一行
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    # 查找所有CSV文件
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 处理每个CSV文件
    processed_count = 0
    error_count = 0
    
    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            # 读取CSV文件
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            # 检查必要的列是否存在
            required_cols = ['SecuCode', 'd1涨跌', 'd2涨跌', 'd3涨跌', 'd4涨跌', 'd5涨跌', '预测值']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"\n警告: {csv_file.name} 缺少必要的列: {missing_cols}")
                error_count += 1
                continue
            
            # 添加预测结果列（如果不存在）
            if '预测结果' not in df.columns:
                df['预测结果'] = None
            
            # 获取所有股票代码
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) < 1:
                print(f"\n警告: {csv_file.name} 中没有股票代码")
                error_count += 1
                continue
            
            target_code = secu_codes[0]  # 第一个是目标点云
            
            print(f"\n处理文件: {csv_file.name}")
            print(f"  目标点云: {target_code}")
            
            # 获取目标点云的数据
            target_rows = df[df['SecuCode'] == target_code]
            
            if len(target_rows) < 5:
                print(f"  警告: 目标点云 {target_code} 的数据不足5行")
                error_count += 1
                continue
            
            # 获取目标点云的行索引
            target_indices = target_rows.index.tolist()
            
            # 对每个交易日进行比较
            correct_count = 0
            total_count = 0
            
            for day in range(1, 6):
                row_idx = target_indices[day - 1]  # 第1行对应d1，第2行对应d2，依此类推
                
                # 获取目标点云该日的实际涨跌值
                actual_col = f'd{day}涨跌'
                actual_value = df.at[row_idx, actual_col]
                
                # 获取预测值
                predicted_value = df.at[row_idx, '预测值']
                
                # 检查是否有缺失值
                if pd.isna(actual_value) or pd.isna(predicted_value):
                    print(f"    第{day}行: 数据缺失 (实际值={actual_value}, 预测值={predicted_value})")
                    df.at[row_idx, '预测结果'] = None
                    continue
                
                # 比较实际值和预测值
                if actual_value == predicted_value:
                    result = 1  # 预测正确
                    correct_count += 1
                else:
                    result = 0  # 预测错误
                
                df.at[row_idx, '预测结果'] = result
                total_count += 1
                
                print(f"    第{day}行 (d{day}): 实际值={actual_value}, 预测值={predicted_value}, 预测结果={result}")
            
            # 计算准确率
            if total_count > 0:
                accuracy = correct_count / total_count * 100
                print(f"  准确率: {correct_count}/{total_count} = {accuracy:.2f}%")
            
            # 保存修改后的文件
            df.to_csv(csv_file, encoding='utf-8-sig', index=False)
            processed_count += 1
            
        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            error_count += 1
            continue
    
    print("\n" + "=" * 80)
    print("处理完成!")
    print(f"成功处理: {processed_count} 个文件")
    if error_count > 0:
        print(f"失败: {error_count} 个文件")
    print("=" * 80)

def evaluate_predictions_with_summary():
    """
    比较目标点云的实际涨跌和预测值（带汇总信息）
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 存储所有文件的统计信息
    all_stats = []
    
    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            required_cols = ['SecuCode', 'd1涨跌', 'd2涨跌', 'd3涨跌', 'd4涨跌', 'd5涨跌', '预测值']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                continue
            
            if '预测结果' not in df.columns:
                df['预测结果'] = None
            
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) < 1:
                continue
            
            target_code = secu_codes[0]
            target_rows = df[df['SecuCode'] == target_code]
            
            if len(target_rows) < 5:
                continue
            
            target_indices = target_rows.index.tolist()
            
            # 对每个交易日进行比较
            correct_by_day = {day: 0 for day in range(1, 6)}
            total_by_day = {day: 0 for day in range(1, 6)}
            
            for day in range(1, 6):
                row_idx = target_indices[day - 1]
                actual_value = df.at[row_idx, f'd{day}涨跌']
                predicted_value = df.at[row_idx, '预测值']
                
                if pd.isna(actual_value) or pd.isna(predicted_value):
                    df.at[row_idx, '预测结果'] = None
                    continue
                
                if actual_value == predicted_value:
                    result = 1
                    correct_by_day[day] += 1
                else:
                    result = 0
                
                df.at[row_idx, '预测结果'] = result
                total_by_day[day] += 1
            
            # 保存文件
            df.to_csv(csv_file, encoding='utf-8-sig', index=False)
            
            # 记录统计信息
            file_stats = {
                'file': csv_file.name,
                'target': target_code
            }
            for day in range(1, 6):
                if total_by_day[day] > 0:
                    accuracy = correct_by_day[day] / total_by_day[day] * 100
                    file_stats[f'd{day}_准确率'] = accuracy
                    file_stats[f'd{day}_正确数'] = correct_by_day[day]
                    file_stats[f'd{day}_总数'] = total_by_day[day]
                else:
                    file_stats[f'd{day}_准确率'] = None
                    file_stats[f'd{day}_正确数'] = 0
                    file_stats[f'd{day}_总数'] = 0
            
            all_stats.append(file_stats)
            
        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            continue
    
    # 生成汇总报告
    print("\n" + "=" * 80)
    print("预测评估汇总报告")
    print("=" * 80)
    
    if all_stats:
        # 创建汇总DataFrame
        summary_df = pd.DataFrame(all_stats)
        
        # 显示整体准确率
        print("\n整体准确率:")
        for day in range(1, 6):
            total_correct = summary_df[f'd{day}_正确数'].sum()
            total_total = summary_df[f'd{day}_总数'].sum()
            if total_total > 0:
                total_accuracy = total_correct / total_total * 100
                print(f"  d{day}: {total_correct}/{total_total} = {total_accuracy:.2f}%")
            else:
                print(f"  d{day}: 没有数据")
        
        # 显示每个文件的准确率（前10个）
        print("\n每个文件的准确率（前10个）:")
        display_cols = ['file', 'target', 'd1_准确率', 'd2_准确率', 'd3_准确率', 'd4_准确率', 'd5_准确率']
        print(summary_df[display_cols].head(10).to_string(index=False))
        
        # 保存汇总报告
        summary_file = input_folder / "预测评估汇总报告.csv"
        summary_df.to_csv(summary_file, encoding='utf-8-sig', index=False)
        print(f"\n汇总报告已保存: {summary_file}")
    
    print("\n" + "=" * 80)
    print("处理完成!")
    print("=" * 80)

def check_evaluation_results():
    """
    检查已处理的文件中的预测结果
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    for csv_file in csv_files[:5]:  # 只检查前5个
        print(f"\n文件: {csv_file.name}")
        
        df = pd.read_csv(csv_file, encoding='utf-8-sig')
        
        if '预测结果' in df.columns and '预测值' in df.columns:
            # 显示目标点云的预测结果
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) > 0:
                target_code = secu_codes[0]
                target_rows = df[df['SecuCode'] == target_code]
                
                if len(target_rows) >= 5:
                    print(f"  目标点云: {target_code}")
                    print(f"  {'交易日':<6} {'实际值':<8} {'预测值':<8} {'预测结果':<8}")
                    print(f"  {'-'*35}")
                    for idx, row in target_rows.iterrows():
                        day = target_rows.index.get_loc(idx) + 1
                        actual = row.get(f'd{day}涨跌', None)
                        predicted = row.get('预测值', None)
                        result = row.get('预测结果', None)
                        print(f"  d{day}:    {actual:<8} {predicted:<8} {result:<8}")
        else:
            print("  没有预测结果或预测值列")

def analyze_accuracy_detailed():
    """
    详细分析预测准确率
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 统计各种情况的次数
    stats = {
        'correct': {day: 0 for day in range(1, 6)},
        'incorrect': {day: 0 for day in range(1, 6)},
        'predict_1': {day: 0 for day in range(1, 6)},
        'predict_0': {day: 0 for day in range(1, 6)},
        'actual_1': {day: 0 for day in range(1, 6)},
        'actual_0': {day: 0 for day in range(1, 6)},
        'correct_actual_1': {day: 0 for day in range(1, 6)},
        'correct_actual_0': {day: 0 for day in range(1, 6)}
    }
    
    for csv_file in tqdm(csv_files, desc="分析文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            if '预测结果' not in df.columns or '预测值' not in df.columns:
                continue
            
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) > 0:
                target_code = secu_codes[0]
                target_rows = df[df['SecuCode'] == target_code]
                
                if len(target_rows) >= 5:
                    for day in range(1, 6):
                        actual_col = f'd{day}涨跌'
                        if actual_col in df.columns:
                            actual_value = target_rows.iloc[day-1][actual_col]
                            predicted_value = target_rows.iloc[day-1]['预测值']
                            result = target_rows.iloc[day-1]['预测结果']
                            
                            if pd.notna(predicted_value) and pd.notna(result):
                                # 统计预测值分布
                                if predicted_value == 1:
                                    stats['predict_1'][day] += 1
                                else:
                                    stats['predict_0'][day] += 1
                                
                                # 统计实际值分布
                                if actual_value == 1:
                                    stats['actual_1'][day] += 1
                                else:
                                    stats['actual_0'][day] += 1
                                
                                # 统计正确/错误
                                if result == 1:
                                    stats['correct'][day] += 1
                                    # 统计正确预测的具体情况
                                    if actual_value == 1:
                                        stats['correct_actual_1'][day] += 1
                                    else:
                                        stats['correct_actual_0'][day] += 1
                                else:
                                    stats['incorrect'][day] += 1
                                
        except Exception as e:
            print(f"\n分析 {csv_file.name} 时出错: {e}")
            continue
    
    # 显示详细统计
    print("\n" + "=" * 80)
    print("详细准确率分析")
    print("=" * 80)
    
    for day in range(1, 6):
        correct = stats['correct'][day]
        incorrect = stats['incorrect'][day]
        total = correct + incorrect
        predict_1 = stats['predict_1'][day]
        predict_0 = stats['predict_0'][day]
        actual_1 = stats['actual_1'][day]
        actual_0 = stats['actual_0'][day]
        correct_actual_1 = stats['correct_actual_1'][day]
        correct_actual_0 = stats['correct_actual_0'][day]
        
        print(f"\nd{day}:")
        print(f"  预测准确率: {correct}/{total} = {correct/total*100:.2f}%" if total > 0 else "  预测准确率: 无数据")
        print(f"  预测值分布: 预测涨(1)={predict_1}, 预测跌(0)={predict_0}")
        print(f"  实际值分布: 实际涨(1)={actual_1}, 实际跌(0)={actual_0}")
        
        if total > 0:
            print(f"  正确预测涨: {correct_actual_1}")
            print(f"  正确预测跌: {correct_actual_0}")
            
            # 计算召回率和精确率
            if actual_1 > 0:
                recall_1 = correct_actual_1 / actual_1 * 100
                print(f"  涨的召回率: {correct_actual_1}/{actual_1} = {recall_1:.2f}%")
            else:
                print(f"  涨的召回率: 无数据")
            
            if predict_1 > 0:
                precision_1 = correct_actual_1 / predict_1 * 100
                print(f"  涨的精确率: {correct_actual_1}/{predict_1} = {precision_1:.2f}%")
            else:
                print(f"  涨的精确率: 无数据")

# ========== 在 Jupyter Notebook 中使用 ==========

def main():
    """主函数 - 标准评估"""
    evaluate_predictions()

def main_with_summary():
    """带汇总信息的评估"""
    evaluate_predictions_with_summary()

def main_check():
    """检查评估结果"""
    check_evaluation_results()

def main_analyze():
    """详细分析准确率"""
    analyze_accuracy_detailed()

# 在 Jupyter 中执行
if __name__ == "__main__":
    # 标准评估
    main()
    
    # 如果需要带汇总信息，取消注释下面的行
    # main_with_summary()
    # main_check()
    # main_analyze()

第14步，生成最终实验报告

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def analyze_prediction_accuracy():
    """
    统计所有CSV文件中的预测结果，按天计算预测准确率
    - 每天有11个目标点云（取决于CSV文件数量）
    - 统计d1到d5每天的预测准确率
    - 结果保存在单独的txt文件中
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    # 查找所有CSV文件
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 初始化统计字典
    stats = {
        'total_files': len(csv_files),
        'by_day': {
            1: {'correct': 0, 'total': 0, 'details': []},
            2: {'correct': 0, 'total': 0, 'details': []},
            3: {'correct': 0, 'total': 0, 'details': []},
            4: {'correct': 0, 'total': 0, 'details': []},
            5: {'correct': 0, 'total': 0, 'details': []}
        },
        'file_details': []
    }
    
    # 处理每个CSV文件
    processed_count = 0
    error_count = 0
    
    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            # 读取CSV文件
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            # 检查必要的列是否存在
            required_cols = ['SecuCode', '预测结果']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"\n警告: {csv_file.name} 缺少必要的列: {missing_cols}")
                error_count += 1
                continue
            
            # 获取所有股票代码
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) < 1:
                print(f"\n警告: {csv_file.name} 中没有股票代码")
                error_count += 1
                continue
            
            target_code = secu_codes[0]  # 第一个是目标点云
            
            # 获取目标点云的数据
            target_rows = df[df['SecuCode'] == target_code]
            
            if len(target_rows) < 5:
                print(f"\n警告: {csv_file.name} 中目标点云数据不足5行")
                error_count += 1
                continue
            
            # 获取目标点云的预测结果
            file_results = {
                'file': csv_file.name,
                'target': target_code,
                'results': {}
            }
            
            for day in range(1, 6):
                # 获取该行的预测结果
                result = target_rows.iloc[day-1]['预测结果']
                
                if pd.isna(result):
                    print(f"\n警告: {csv_file.name} 第{day}天预测结果为空")
                    continue
                
                # 统计
                stats['by_day'][day]['total'] += 1
                if result == 1:
                    stats['by_day'][day]['correct'] += 1
                    file_results['results'][f'd{day}'] = 1
                else:
                    file_results['results'][f'd{day}'] = 0
            
            # 记录文件详细信息
            stats['file_details'].append(file_results)
            processed_count += 1
            
        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            error_count += 1
            continue
    
    print("\n" + "=" * 80)
    print(f"成功处理: {processed_count} 个文件")
    if error_count > 0:
        print(f"失败: {error_count} 个文件")
    print("=" * 80)
    
    # 生成统计报告
    generate_report(stats, input_folder)
    
    return stats

def generate_report(stats, output_folder):
    """
    生成统计报告并保存为txt文件
    """
    # 创建报告内容
    report_lines = []
    
    # 标题
    report_lines.append("=" * 80)
    report_lines.append("预测结果统计分析报告")
    report_lines.append("=" * 80)
    report_lines.append(f"分析日期: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"分析文件总数: {stats['total_files']}")
    report_lines.append(f"成功处理文件数: {len(stats['file_details'])}")
    report_lines.append("")
    
    # 按天统计
    report_lines.append("-" * 80)
    report_lines.append("按天统计预测准确率")
    report_lines.append("-" * 80)
    report_lines.append(f"{'交易日':<8} {'正确数':<10} {'总数':<10} {'准确率':<10}")
    report_lines.append("-" * 80)
    
    for day in range(1, 6):
        correct = stats['by_day'][day]['correct']
        total = stats['by_day'][day]['total']
        if total > 0:
            accuracy = correct / total * 100
            report_lines.append(f"d{day}:     {correct:<10} {total:<10} {accuracy:.2f}%")
        else:
            report_lines.append(f"d{day}:     {correct:<10} {total:<10} 无数据")
    
    report_lines.append("-" * 80)
    
    # 整体统计
    total_correct = sum(stats['by_day'][day]['correct'] for day in range(1, 6))
    total_total = sum(stats['by_day'][day]['total'] for day in range(1, 6))
    if total_total > 0:
        total_accuracy = total_correct / total_total * 100
        report_lines.append(f"整体:     {total_correct:<10} {total_total:<10} {total_accuracy:.2f}%")
    else:
        report_lines.append(f"整体:     {total_correct:<10} {total_total:<10} 无数据")
    
    report_lines.append("=" * 80)
    report_lines.append("")
    
    # 每个文件的详细结果
    report_lines.append("-" * 80)
    report_lines.append("每个文件的详细预测结果")
    report_lines.append("-" * 80)
    report_lines.append(f"{'序号':<6} {'文件名':<40} {'d1':<6} {'d2':<6} {'d3':<6} {'d4':<6} {'d5':<6}")
    report_lines.append("-" * 80)
    
    for idx, file_detail in enumerate(stats['file_details'], 1):
        file_name = file_detail['file'][:38]  # 截断文件名
        results = file_detail['results']
        d1 = results.get('d1', 'N/A')
        d2 = results.get('d2', 'N/A')
        d3 = results.get('d3', 'N/A')
        d4 = results.get('d4', 'N/A')
        d5 = results.get('d5', 'N/A')
        report_lines.append(f"{idx:<6} {file_name:<40} {d1:<6} {d2:<6} {d3:<6} {d4:<6} {d5:<6}")
    
    report_lines.append("-" * 80)
    report_lines.append("")
    
    # 混淆矩阵统计（针对每个交易日）
    report_lines.append("-" * 80)
    report_lines.append("各交易日详细统计")
    report_lines.append("-" * 80)
    
    for day in range(1, 6):
        correct = stats['by_day'][day]['correct']
        total = stats['by_day'][day]['total']
        if total > 0:
            accuracy = correct / total * 100
            incorrect = total - correct
            report_lines.append(f"d{day}:")
            report_lines.append(f"  正确预测: {correct}")
            report_lines.append(f"  错误预测: {incorrect}")
            report_lines.append(f"  总预测数: {total}")
            report_lines.append(f"  准确率: {accuracy:.2f}%")
            report_lines.append("")
        else:
            report_lines.append(f"d{day}: 没有预测数据")
            report_lines.append("")
    
    report_lines.append("=" * 80)
    report_lines.append("报告生成完成")
    report_lines.append("=" * 80)
    
    # 保存报告到txt文件
    report_file = output_folder / "预测结果统计分析报告.txt"
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))
    
    print(f"\n报告已保存: {report_file}")
    
    # 打印报告到控制台
    print("\n" + "=" * 80)
    print("预测结果统计分析报告")
    print("=" * 80)
    print(f"分析文件总数: {stats['total_files']}")
    print(f"成功处理文件数: {len(stats['file_details'])}")
    print("\n按天统计预测准确率:")
    print(f"{'交易日':<8} {'正确数':<10} {'总数':<10} {'准确率':<10}")
    print("-" * 40)
    for day in range(1, 6):
        correct = stats['by_day'][day]['correct']
        total = stats['by_day'][day]['total']
        if total > 0:
            accuracy = correct / total * 100
            print(f"d{day}:     {correct:<10} {total:<10} {accuracy:.2f}%")
        else:
            print(f"d{day}:     {correct:<10} {total:<10} 无数据")
    print("-" * 40)
    total_correct = sum(stats['by_day'][day]['correct'] for day in range(1, 6))
    total_total = sum(stats['by_day'][day]['total'] for day in range(1, 6))
    if total_total > 0:
        print(f"整体:     {total_correct:<10} {total_total:<10} {total_accuracy:.2f}%")
    print("=" * 80)

def analyze_with_visualization():
    """
    分析预测准确率并生成简单的可视化（可选）
    """
    import matplotlib.pyplot as plt
    
    # 获取统计数据
    stats = analyze_prediction_accuracy()
    
    if stats and 'by_day' in stats:
        # 准备数据
        days = [f'd{day}' for day in range(1, 6)]
        correct = [stats['by_day'][day]['correct'] for day in range(1, 6)]
        total = [stats['by_day'][day]['total'] for day in range(1, 6)]
        accuracy = [(correct[i] / total[i] * 100) if total[i] > 0 else 0 for i in range(5)]
        
        # 创建图表
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # 柱状图：正确vs错误
        ax1.bar(days, correct, label='正确', color='green', alpha=0.7)
        incorrect = [total[i] - correct[i] for i in range(5)]
        ax1.bar(days, incorrect, bottom=correct, label='错误', color='red', alpha=0.7)
        ax1.set_xlabel('交易日')
        ax1.set_ylabel('预测数量')
        ax1.set_title('各交易日预测结果分布')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 柱状图：准确率
        ax2.bar(days, accuracy, color='blue', alpha=0.7)
        ax2.set_xlabel('交易日')
        ax2.set_ylabel('准确率 (%)')
        ax2.set_title('各交易日预测准确率')
        ax2.set_ylim(0, 100)
        ax2.grid(True, alpha=0.3)
        
        # 在柱子上显示数值
        for i, (day, acc) in enumerate(zip(days, accuracy)):
            ax2.text(i, acc + 1, f'{acc:.1f}%', ha='center', va='bottom')
        
        plt.tight_layout()
        
        # 保存图表
        input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
        plt.savefig(input_folder / '预测准确率分析图.png', dpi=300, bbox_inches='tight')
        print(f"\n图表已保存: {input_folder / '预测准确率分析图.png'}")
        
        plt.show()

def generate_summary_report():
    """
    生成汇总报告，包含所有文件的预测结果摘要
    """
    # 输入文件夹路径 - 修改为您指定的路径
    input_folder = Path(r"D:\量化\我自己的实验\3rd test\筛选结果\学生操作数据汇总\20240628_select\处理结果\processed_results")
    
    if not input_folder.exists():
        print(f"错误: 文件夹不存在: {input_folder}")
        return
    
    # 查找所有CSV文件
    csv_files = list(input_folder.glob("*.csv"))
    
    if not csv_files:
        print(f"错误: 在 {input_folder} 中没有找到CSV文件")
        return
    
    print(f"找到 {len(csv_files)} 个CSV文件")
    print("=" * 80)
    
    # 收集所有文件的预测结果
    all_results = []
    
    for csv_file in tqdm(csv_files, desc="处理CSV文件", unit="个"):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8-sig')
            
            if '预测结果' not in df.columns:
                continue
            
            secu_codes = df['SecuCode'].unique()
            if len(secu_codes) < 1:
                continue
            
            target_code = secu_codes[0]
            target_rows = df[df['SecuCode'] == target_code]
            
            if len(target_rows) < 5:
                continue
            
            # 收集该文件的结果
            result_dict = {
                'file': csv_file.name,
                'target': target_code
            }
            
            for day in range(1, 6):
                result = target_rows.iloc[day-1]['预测结果']
                result_dict[f'd{day}'] = result if pd.notna(result) else 'N/A'
            
            all_results.append(result_dict)
            
        except Exception as e:
            print(f"\n处理 {csv_file.name} 时出错: {e}")
            continue
    
    # 创建汇总DataFrame
    if all_results:
        summary_df = pd.DataFrame(all_results)
        
        # 保存汇总报告
        summary_file = input_folder / "预测结果汇总报告.csv"
        summary_df.to_csv(summary_file, encoding='utf-8-sig', index=False)
        print(f"\n汇总报告已保存: {summary_file}")
        
        # 显示前10行
        print("\n汇总报告预览（前10行）:")
        print(summary_df.head(10).to_string(index=False))
        
        # 统计各天的预测分布
        print("\n" + "=" * 80)
        print("预测结果分布统计")
        print("=" * 80)
        for day in range(1, 6):
            col = f'd{day}'
            if col in summary_df.columns:
                values = summary_df[col].value_counts()
                print(f"\n{col}:")
                for value, count in values.items():
                    label = '正确(1)' if value == 1 else '错误(0)' if value == 0 else '无数据'
                    print(f"  {label}: {count} 个文件 ({count/len(summary_df)*100:.1f}%)")
        
        return summary_df
    else:
        print("没有找到有效的预测结果")
        return None

# ========== 在 Jupyter Notebook 中使用 ==========

def main():
    """主函数 - 统计分析"""
    stats = analyze_prediction_accuracy()
    return stats

def main_with_visualization():
    """带可视化的统计分析"""
    analyze_with_visualization()

def main_summary():
    """生成汇总报告"""
    generate_summary_report()

# 在 Jupyter 中执行
if __name__ == "__main__":
    # 标准统计分析
    main()
    
    # 如果需要可视化，取消注释下面的行
    # main_with_visualization()
    
    # 如果需要生成汇总报告，取消注释下面的行
    # main_summary()